# Driver fatigue from blink sequences — presentation demonstration

A run-only prototype of the deliverable from the main project notebook. It loads the
trained checkpoint and the pre-cut demo clips and serves the dashboard.

**It does not train, cross-validate, extract the corpus, or reproduce any result.** Those
take several hours and belong in the full notebook. Everything here is inference.

The full pipeline is reproduced verbatim — the same geometry, the same blink retrieval,
the same normalisation and the same model class — because the demo has to be the pipeline
the numbers were measured on, not a simplified re-implementation of it.

### Requires, from a completed run of the main notebook

| Artefact | Location | Made by |
|---|---|---|
| `deliverable_blink_n4.pt` | `artifacts/models/` | §9 |
| `s*_*.mp4` demo clips, `s*_enrol_*.mp4` enrolment clips | `artifacts/demo_clips/` | §12.1 |
| `annotated_*.mp4` overlays *(optional)* | `artifacts/demo_clips/annotated/` | §12.1 |
| `invalid_*.mp4` refusal exhibits *(optional)* | `artifacts/demo_clips/` | §12.1 |

Without the invalid clips the refusal path cannot be demonstrated. The overlays are
optional because a live run now writes its own.

### How to run it

Run every cell in order, top to bottom. Set `PROJECT` in §2 to the Drive folder the full
notebook wrote to. Four cells report something that must be right before the app is worth
launching, and each says so in its output:

| Cell | Confirms |
|---|---|
| §2 preflight | the checkpoint and the clips are where `PROJECT` says |
| §3 blink gate | `source: checkpoint`, closing 0.75 and reopening 0.85 |
| §9 smoke test | one clip scores end to end, with a blink count in the dozens |
| §11 agreement check | the live geometry equals `frame_geometry`, and what a reduced detection width would cost |

The last cell prints a local URL, a public link, and a frame-stream URL. Open the public
link if the demo is being screen-shared or shown from another machine. The server runs in
the background, so the cell finishes; re-running it restarts the app.

Cold start is a few minutes: install, mount Drive, download the MediaPipe landmarker,
cache the enrolment calibrations, score one clip, launch.

### What to show, in order

1. **Live**, on a drowsy clip with enrolment. The warm-up is the argument: no verdict
   exists until 30 blink events do. Talk over it — the blink counter, the EAR trace and
   the feature table are all moving while the score is still absent.
2. The same clip on **Summary** with the enrolment box unticked. The out-of-protocol
   banner appears and the verdict can change. This is the deployment limitation.
3. A `[refusal demo]` clip. The system declines and says why.
4. **Rendered overlay**, already filled in by step 1: the chain from mesh to blink to
   features to score.

## 1 · Environment

In [ ]:
%pip install -q mediapipe scipy pandas numpy matplotlib tqdm gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 15.9 MB/s eta 0:00:00


In [ ]:
try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False

import re, json, math, time, random, shutil, zipfile, subprocess, threading

from pathlib import Path

import numpy as np

import pandas as pd

import cv2

import torch

import torch.nn as nn

import matplotlib.pyplot as plt

from tqdm.auto import tqdm

# =============================================================================
# 7.4  blink events - the model input used by the benchmark
# =============================================================================
# Ghoddoosian et al. (2019) feed the network a sequence of 30 consecutive blink
# events with a stride of 2, which is 2-3 minutes of recording per training example.
# A fixed 10 s window (appendix A2) contains only a handful of blinks and often no
# drowsiness-relevant event at all.
#
# Ported from the reference implementation: the Blink Retrieval Algorithm (Alg. 1),
# which splits one detected closure into the several quick blinks it may contain, and
# the four blink features of Eq. 2-5 (duration, amplitude, opening velocity, frequency).
#
# Substituted: their candidate detector, an SVM over raw dlib EAR. Raw EAR has no
# consistent scale across people, so no single threshold means "closed" for everyone.
# ear_cal is already divided by the subject's open-eye reference (stage A, §7.0), so a
# fixed hysteresis threshold is meaningful here. §7.5 validates the substitution.
#
# Every feature below reads ear_cal, so all four depend on the stage-A reference:
# amplitude and velocity through the arithmetic, duration and frequency through which
# closures the gate fires on. The alternative reference is examined in §8.3 and A3.
#
# The retrieval threshold is 0.6*max + 0.4*min per segment, so it is scale-invariant and
# transfers from dlib EAR to MediaPipe EAR unchanged.
from scipy.signal import medfilt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", DEVICE)

device: cuda


## 2 · Paths and preflight

`PROJECT` must point at the same Drive folder the full notebook wrote to.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/msc-datascience/msc-project/fatigue_project")
else:
    PROJECT = Path("./fatigue_project")

ARTIFACTS     = PROJECT / "artifacts"

MODELS_DIR    = ARTIFACTS / "models"

DEMO_DIR      = ARTIFACTS / "demo_clips"

CKPT_PATH = MODELS_DIR / "deliverable_blink_n4.pt"

for _d in (ARTIFACTS, MODELS_DIR, DEMO_DIR):
    _d.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
# ---- preflight: this notebook loads artefacts, it never trains ---------------
# Everything below is inference. Missing artefacts are reported here, by name.
_missing, _warn = [], []

if not CKPT_PATH.exists():
    _missing.append(f"trained checkpoint  ->  {CKPT_PATH}")

_clips = sorted(DEMO_DIR.glob("s*_*.mp4")) if DEMO_DIR.exists() else []
_demo  = [p for p in _clips if "_enrol_" not in p.stem and "annotated_" not in p.stem]
_enrol = [p for p in _clips if "_enrol_" in p.stem]
_bad   = sorted(DEMO_DIR.glob("invalid_*.mp4")) if DEMO_DIR.exists() else []
_annot = sorted((DEMO_DIR / "annotated").glob("annotated_*.mp4")) \
         if (DEMO_DIR / "annotated").exists() else []

if not _demo:
    _missing.append(f"demo clips (s*_*.mp4)  ->  {DEMO_DIR}")
if not _enrol:
    _warn.append("no enrolment clips (s*_enrol_*.mp4): every clip will be scored under the "
                 "population fallback, which is the out-of-protocol condition.")
if not _annot:
    _warn.append("no pre-rendered overlays in demo_clips/annotated: the 'Show annotated "
                 "video' button will render on demand, which takes a few minutes per clip.")
if not _bad:
    _warn.append("no invalid_*.mp4 clips: the refusal path will not be demonstrable.")

if _missing:
    raise FileNotFoundError(
        "Demo artefacts not found. This notebook does not create them - run the full "
        "notebook's \u00a79 (checkpoint) and \u00a712.1 (demo clips) once, then point PROJECT at "
        "the same Drive folder.\n  missing:\n    " + "\n    ".join(_missing))

print(f"checkpoint : {CKPT_PATH.name} ({CKPT_PATH.stat().st_size/1e6:.1f} MB)")
print(f"demo clips : {len(_demo)} scoreable, {len(_enrol)} enrolment, {len(_bad)} refusal")
print(f"overlays   : {len(_annot)} pre-rendered")
for w in _warn:
    print("  note:", w)

checkpoint : deliverable_blink_n4.pt (0.2 MB)
demo clips : 18 scoreable, 5 enrolment, 1 refusal
overlays   : 16 pre-rendered


## 3 · Configuration

The operating point the checkpoint was trained under. `EVENT_CLOSED_T` is the one that
matters most here: it is the candidate gate calibrated in §7.0b, and changing it would
mean the clips are no longer scored the way the model was fitted.

In [ ]:
CONFIG = dict(
    # --- extraction: every source frame, for clean blink dynamics ---
    SAMPLE_EVERY          = 1,        # keep every source frame (~30 fps at source)
    MAX_FRAMES_PER_VIDEO  = 20000,    # ~11 min at 30 fps - never binds on 10-min videos
    N_EXTRACT_THREADS     = 8,        # one worker thread per video; each has its own landmarker
    # --- windowing: time-based, resampled to a fixed grid ---
    TARGET_HZ             = 15,       # every window is resampled to this rate (fps-agnostic)
    WIN_SECONDS           = 10,       # window length in seconds (validation-selected in §8.1)
    WIN_OVERLAP           = 0.66,     # 2/3 overlap between consecutive windows
    SEQ_FEATURES          = ["ear_cal", "mar", "pitch", "yaw", "roll", "head_drop"],
    # --- per-subject normalisation: stage B, Ghoddoosian-style calibration ---
    PER_SUBJECT_NORM      = True,     # z-score each channel vs the subject's own alert baseline
    CALIB_FRACTION        = 1/3,      # frame-window track only (appendix A2); the blink pipeline
                                      # uses CALIB_FRACTION_BLINKS and CALIB_END in §8
    #                                   (Ghoddoosian et al. 2019; held out of windowing)
    # --- labels ---
    STATES                = {0: "alert", 5: "lowvig", 10: "drowsy"},
    BINARY_STATES         = (0, 10),  # primary task: alert vs drowsy
    # --- split (by subject) ---
    VAL_SUBJECT_FRAC      = 0.15,
    TEST_SUBJECT_FRAC     = 0.20,
    # --- the PERCLOS *feature* (graded lid droop; validated by the §8.3 sweep) ---
    PERCLOS_T             = 0.80,     # ear_cal < 0.80 == eye >=20% closed (literature "p20")
    # --- event labelling only (§7.0b): a far stricter test, with hysteresis ---
    EVENT_CLOSED_T        = 0.25,
    EVENT_OPEN_T          = 0.35,
    EVENT_MIN_BLINK_F     = 2,
    EVENT_MIN_OPEN_F      = 2,
    MICROSLEEP_MIN_S      = 0.5,
    YAWN_MAR_T            = 0.60,
    YAWN_MIN_S            = 1.5,
    # --- geometry corrections ---
    PITCH_UNWRAP          = True,
    # --- demo clips for Gradio (feature-driven selection, §12.2) ---
    N_DEMO_SUBJECTS       = 6,
    # 180 s, not 30 s. One model input is 30 blink events, which is 2-3 min of video.
    # At the 13-21 blinks/min measured in §7.5 a 30 s clip yields ~7 blinks, so most of
    # the sequence would be padding. This is a property of the method, written up in §12.
    DEMO_CLIP_LEN_S       = 180,
    DEMO_CLIPS_PER_VIDEO  = 3,
    DEMO_SCORE_HOP_S      = 5,
    DEMO_MIN_FACE_COV     = 0.95,
    DEMO_CLIP_START_S     = 120,
)

In [ ]:
BLINK_PAD_S      = 0.20     # seconds of context either side of a detected closure

BLINK_MEDFILT    = 3        # median filter width (odd), noise suppression before Alg.1

BLINK_SEQ_LEN    = 30       # T, blinks per sequence (Ghoddoosian Fig. 4a)

BLINK_SEQ_STRIDE = 2        # sliding stride over blinks

FEATS_PUB = ["frequency_pub", "amplitude", "duration_pub", "velocity_pub"]

FEATS_COR = ["frequency_cor", "amplitude", "duration_cor", "velocity_cor"]

DISCRETISE_STEP = 3.34
CUT_LO, CUT_HI = DISCRETISE_STEP, 2 * DISCRETISE_STEP  # operational boundaries: 3.34, 6.68

LABELS_POOL  = np.array([0.0, 5.0, 10.0])

### The blink gate

`CONFIG` above carries a placeholder (`EVENT_CLOSED_T = 0.25`); the calibrated gate is
0.75, closing above 0.85. The difference matters: the calibrated gate yields about twelve
candidate closures a minute against roughly one at the placeholder, and a clip needs 30
blink events before a single sequence can be scored.

The gate is therefore read from the checkpoint, which is where a property of the trained
deliverable belongs. The first cell writes it there if it is absent; the second applies
it. Both are safe to re-run.

In [ ]:
# ---- detector operating point: keep the demo checkpoint read-only -------------
# The original version patched the checkpoint in place. That makes a supposedly
# inference-only notebook mutate its model artefact on Drive and can fail on a synced or
# read-only file. We only inspect it here; cell 12 applies the documented fallback in memory.
ADOPTED_CLOSED_T, ADOPTED_OPEN_T = 0.75, 0.85

_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
if _ck.get("event_closed_t") is not None and _ck.get("event_open_t") is not None:
    print(f"checkpoint carries the gate: close {_ck['event_closed_t']:.2f}, "
          f"reopen {_ck['event_open_t']:.2f}")
else:
    print("checkpoint does not carry the blink gate; the demo will use the documented "
          f"in-memory fallback {ADOPTED_CLOSED_T:.2f}/{ADOPTED_OPEN_T:.2f}. "
          "The checkpoint file is not modified.")
del _ck


checkpoint carries the gate: close 0.75, reopen 0.85


In [ ]:
# ---- operating point: taken from the checkpoint, not from CONFIG's defaults ----
# CONFIG above ships the pre-calibration placeholder. The gate that produced the training
# features is chosen by the sweep in 7.0b and stored in the checkpoint, so training and
# inference share one operating point.
_hdr = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
if _hdr.get("event_closed_t") is not None and _hdr.get("event_open_t") is not None:
    CONFIG["EVENT_CLOSED_T"] = float(_hdr["event_closed_t"])
    CONFIG["EVENT_OPEN_T"]   = float(_hdr["event_open_t"])
    _gate_src = "checkpoint"
else:
    CONFIG["EVENT_CLOSED_T"], CONFIG["EVENT_OPEN_T"] = 0.75, 0.85
    _gate_src = ("documented default - this checkpoint does not carry the gate. Run the "
                 "patch cell above; the demo still works meanwhile.")
del _hdr

print(f"blink gate: close below {CONFIG['EVENT_CLOSED_T']:.2f}, "
      f"reopen above {CONFIG['EVENT_OPEN_T']:.2f}")
print(f"  source: {_gate_src}")
if CONFIG["EVENT_CLOSED_T"] < 0.5:
    print("  WARNING: this gate is far stricter than the calibrated one. Expect very few "
          "blinks and a refusal.")

blink gate: close below 0.75, reopen above 0.85
  source: checkpoint


## 4 · Face landmarks and geometry

MediaPipe Face Mesh, then EAR, MAR and head pose per frame. One landmarker per thread —
a shared instance is not safe under concurrent `detect()`.

In [ ]:
import os, urllib.request

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

LM_MODEL = "face_landmarker.task"
if not (os.path.exists(LM_MODEL) and os.path.getsize(LM_MODEL) > 1_000_000):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
        "face_landmarker/float16/1/face_landmarker.task", LM_MODEL)
print(f"{LM_MODEL}: {os.path.getsize(LM_MODEL)/1e6:.1f} MB ok")

face_landmarker.task: 3.8 MB ok


In [ ]:
def new_landmarker():
    return vision.FaceLandmarker.create_from_options(
        vision.FaceLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=LM_MODEL),
            num_faces=1))

# one landmarker per worker thread (a shared instance is not safe under concurrent detect())
_tls = threading.local()

def thread_landmarker():
    lm = getattr(_tls, "lm", None)
    if lm is None:
        lm = new_landmarker(); _tls.lm = lm
    return lm

In [ ]:
# ---- landmark indices (MediaPipe 468-point mesh) ----
LEFT_EYE  = [33, 160, 158, 133, 153, 144]   # p1..p6 around the left eye

RIGHT_EYE = [362, 385, 387, 263, 373, 380]

MOUTH     = [61, 291, 13, 14]               # corners + upper/lower lip midpoints

# geometry helpers. Names are spelled out because the earlier short names (_d, _ear)
# were silently rebound by a later cell; the self-test below catches a recurrence.
_lm_dist = lambda a, b: math.hypot(a[0] - b[0], a[1] - b[1])

_lm_ear  = lambda p: (_lm_dist(p[1], p[5]) + _lm_dist(p[2], p[4])) / (2 * _lm_dist(p[0], p[3]) + 1e-6)

_lm_mar  = lambda p: _lm_dist(p[2], p[3]) / (_lm_dist(p[0], p[1]) + 1e-6)

# 6 canonical 3D face points (generic head model, mm) matched to mesh indices for solvePnP
FACE_3D = np.array([
    (   0.0,    0.0,    0.0),   # nose tip        -> 1
    (   0.0, -330.0,  -65.0),   # chin            -> 152
    (-225.0,  170.0, -135.0),   # left eye outer  -> 33
    ( 225.0,  170.0, -135.0),   # right eye outer -> 263
    (-150.0, -150.0, -125.0),   # mouth left      -> 61
    ( 150.0, -150.0, -125.0),   # mouth right     -> 291
], dtype=np.float64)

POSE_IDX = [1, 152, 33, 263, 61, 291]

def head_pose(lm, w, h):
    # returns (pitch, yaw, roll) in degrees, plus rvec/tvec for axis drawing
    pts2d = np.array([(lm[i].x * w, lm[i].y * h) for i in POSE_IDX], dtype=np.float64)
    cam   = np.array([[w, 0, w/2], [0, w, h/2], [0, 0, 1]], dtype=np.float64)
    ok, rvec, tvec = cv2.solvePnP(FACE_3D, pts2d, cam, np.zeros((4, 1)),
                                  flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok:
        return (np.nan, np.nan, np.nan), None, None, cam
    R, _ = cv2.Rodrigues(rvec)
    ang, _, _, _, _, _ = cv2.RQDecomp3x3(R)
    return (ang[0], ang[1], ang[2]), rvec, tvec, cam

def frame_geometry(frame_bgr, lm_engine):
    # -> dict of the six per-frame features, or None if no face found
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = lm_engine.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.face_landmarks:
        return None
    h, w = frame_bgr.shape[:2]
    lm = res.face_landmarks[0]
    P = lambda i: (lm[i].x * w, lm[i].y * h)
    ear = (_lm_ear([P(i) for i in LEFT_EYE]) + _lm_ear([P(i) for i in RIGHT_EYE])) / 2
    mar = _lm_mar([P(i) for i in MOUTH])
    head_drop = (P(1)[1] - (P(33)[1] + P(263)[1]) / 2) / h
    (pitch, yaw, roll), _, _, _ = head_pose(lm, w, h)
    return dict(ear=ear, mar=mar, head_drop=head_drop, pitch=pitch, yaw=yaw, roll=roll)

## 5 · Video to per-frame geometry

`detect_rotation` handles the sideways recordings; `prepare_video` is stage A of the three
normalisation stages — it divides EAR by an open-eye reference. At serve time that
reference comes from the driver's enrolment clip, or from population statistics when there
is none.

In [ ]:
# Some videos (fold-5 s49) are stored sideways, which breaks face detection. Probe a
# few frames in all four orientations and keep the one that finds a face. None = upright.
ROTATIONS = [None, cv2.ROTATE_90_CLOCKWISE, cv2.ROTATE_90_COUNTERCLOCKWISE, cv2.ROTATE_180]

def detect_rotation(video_path, probe_times=(20, 60, 100)):
    lm = thread_landmarker()
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    scores = [0] * len(ROTATIONS)
    for t in probe_times:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(t * fps))
        ok, fr = cap.read()
        if not ok: continue
        for i, r in enumerate(ROTATIONS):
            f2 = fr if r is None else cv2.rotate(fr, r)
            rgb = cv2.cvtColor(f2, cv2.COLOR_BGR2RGB)
            if lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)).face_landmarks:
                scores[i] += 1
    cap.release()
    best = int(np.argmax(scores))
    return ROTATIONS[best] if scores[best] > 0 else None

def extract_video(video_path, sample_every, max_frames, progress=False):
    # progress=True gives a per-frame bar - use it for single-video calls only, not inside
    # the thread pool where the per-video bar is the right unit.
    rot = detect_rotation(video_path)            # None for upright videos
    lm = thread_landmarker()                     # this thread's own landmarker
    cap = cv2.VideoCapture(str(video_path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    kept_total = min(max_frames, n_total // sample_every) if n_total > 0 else None
    bar = tqdm(total=kept_total, unit="frame", leave=False, disable=not progress,
               desc=f"  {Path(video_path).name[:24]}")
    rows, idx, kept = [], 0, 0
    while kept < max_frames:
        ok, frame = cap.read()
        if not ok: break
        if idx % sample_every == 0:
            if rot is not None:
                frame = cv2.rotate(frame, rot)
            g = frame_geometry(frame, lm)
            if g is None:
                rows.append(dict(frame=kept, face=0, ear=np.nan, mar=np.nan,
                                 head_drop=np.nan, pitch=np.nan, yaw=np.nan, roll=np.nan))
            else:
                rows.append(dict(frame=kept, face=1, **g))
            kept += 1
            bar.update(1)
        idx += 1
    bar.close(); cap.release()
    df = pd.DataFrame(rows)
    df["src_fps"] = src_fps
    return df

In [ ]:
def unwrap_pitch(p):
    """Rotate the Euler branch cut by 180 deg so a frontal face sits near 0, not +/-180.
    +177 -> -3 and -173 -> +7, i.e. 10 deg apart as they physically are. NaN-safe."""
    return ((p + 360.0) % 360.0) - 180.0

def prepare_video(df, ear_ref=None):
    """Sort, unwrap pitch, scale EAR by an open-eye reference, impute gaps.

    Stage A of the three normalisation stages: per frame, and by default per video.
    Not to be confused with stage B (§8), which z-scores blink-event features against
    a third of the subject's own alert blinks.

    `ear_ref=None` takes the P90 of EAR over this video's own frames. The pipeline
    passes an explicit reference instead: the P90 of that subject's alert recording,
    applied to all three of their recordings.

    The argument exists because per-recording referencing is label-conditional - the
    open-eye value is estimated from the recording being classified, so a session in
    which the lids sit low throughout has part of that effect divided out before the
    detector runs (§8.3 measures the shift, appendix A3 compares the two settings).
    It is also clip-length dependent, which matters at serving time.
    """
    df = df.sort_values("frame").reset_index(drop=True).copy()
    if CONFIG["PITCH_UNWRAP"]:
        df["pitch"] = unwrap_pitch(df["pitch"])
    if ear_ref is None:
        ear_ref = np.nanpercentile(df["ear"], 90) if df["ear"].notna().any() else 1.0
    ref = float(ear_ref) if float(ear_ref) > 1e-6 else 1e-6
    df["ear_cal"] = df["ear"] / ref
    for c in ["ear_cal", "mar", "head_drop", "pitch", "yaw", "roll"]:
        df[c] = df[c].interpolate(limit_direction="both")
    df[["pitch", "yaw", "roll"]] = df[["pitch", "yaw", "roll"]].fillna(0.0)
    df[["ear_cal", "mar", "head_drop"]] = df[["ear_cal", "mar", "head_drop"]].fillna(
        df[["ear_cal", "mar", "head_drop"]].median())
    df["eff_fps"] = df["src_fps"] / CONFIG["SAMPLE_EVERY"]
    df.attrs["ear_ref"] = ref      # so callers can reuse this subject's reference
    return df

## 6 · Blink detection

The hysteresis gate proposes candidate closures; the Blink Retrieval Algorithm
(Ghoddoosian et al. 2019, Alg. 1) splits a candidate into the blinks it contains, and
Eq. 2–5 give the four features the model consumes.

In [ ]:
def closure_runs_hysteresis(ear_cal, t_close=None, t_open=None,
                            min_f=None, min_open_f=None):
    """Schmitt-triggered closure detection -> list of (start_idx, end_idx_exclusive).

    Two independent guards against landmark jitter:
      1. hysteresis    - enter closed below `t_close`, leave only above `t_open`;
      2. exit debounce - the eye must stay above `t_open` for `min_open_f` consecutive
         frames before the closure is deemed over, so a single bad frame mid-closure
         cannot split one long closure into two short ones.
    Closures shorter than `min_f` frames are discarded."""
    t_close    = CONFIG["EVENT_CLOSED_T"]    if t_close    is None else t_close
    t_open     = CONFIG["EVENT_OPEN_T"]      if t_open     is None else t_open
    min_f      = CONFIG["EVENT_MIN_BLINK_F"] if min_f      is None else min_f
    min_open_f = CONFIG["EVENT_MIN_OPEN_F"]  if min_open_f is None else min_open_f
    runs, start, closed, open_run = [], None, False, 0
    for i, v in enumerate(ear_cal):
        if not closed:
            if v < t_close: closed, start, open_run = True, i, 0
        else:
            if v > t_open:
                open_run += 1
                if open_run >= min_open_f:                 # confirmed re-opening
                    end = i - open_run + 1
                    if end - start >= min_f: runs.append((start, end))
                    closed, start, open_run = False, None, 0
            else:
                open_run = 0                               # spike was noise; still closed
    if closed and len(ear_cal) - start >= min_f:
        runs.append((start, len(ear_cal)))
    return runs

In [ ]:
def blink_retrieval(x, epsilon=0.01):
    """Ghoddoosian et al. (2019) supplementary Algorithm 1, verbatim.

    Input : x, the EAR values of one candidate closure (plus padding context).
    Output: list of (start, bottom, end) indices into x, one per retrieved blink.

    The algorithm finds local extrema via derivative sign changes, labels each
    extremum above/below a per-segment threshold, and reads off each downward-
    then-upward excursion as one blink. Runs in O(M).
    """
    x = np.asarray(x, dtype=np.float64)
    M = len(x)
    if M < 3:
        return []

    d = np.diff(x)                                        # step 1
    if d[0] == 0:                                         # step 2
        d[0] = -epsilon
    for n in range(1, len(d)):                            # step 3
        if d[n] == 0:
            d[n] = d[n - 1] * epsilon

    c = d[1:] * d[:-1]                                    # step 4
    ext = [n + 1 for n in range(len(c)) if c[n] < 0]      # step 7: interior extrema
    e = [0] + ext + [M - 1]                               # steps 5-7

    THR = 0.6 * float(np.max(x)) + 0.4 * float(np.min(x)) # step 8
    t = [1] + [1 if x[i] > THR else -1 for i in ext] + [1]   # steps 9-11

    z = [t[n + 1] * t[n] for n in range(len(t) - 1)]      # step 12
    s = [n for n, v in enumerate(z) if v < 0]             # step 13
    if len(s) % 2 == 1:                                   # unbalanced crossing at an edge
        s = s[:-1]

    blinks = []
    for i in range(len(s) // 2):                          # steps 14-18
        a, b = s[2 * i], s[2 * i + 1]
        if b + 1 >= len(e):
            continue
        start, bottom, end = e[a], e[b], e[b + 1]
        if end > start:
            blinks.append((start, bottom, end))
    return blinks

def extract_blinks_for_video(vdf, eff_fps, closure_runs_fn=None):
    """Per-frame geometry for one video -> a DataFrame of blink events.

    Candidate closures come from §7.0b's hysteresis detector; each candidate is
    padded with context and passed to blink_retrieval, which may split it into
    several blinks. Returns one row per blink with both feature variants.

    Feature variants (an experiment, not a correction applied silently):
      published  duration in frames, frequency as 100*blinks/frames (Eq. 2, 5)
      corrected  duration in seconds, frequency in blinks per minute
    The published definitions are frame-rate dependent and UTA-RLDD runs at 12-30 fps,
    so a 300 ms blink is 4 frames at 12 fps and 9 at 30 fps. Amplitude is a ratio of
    EAR values and is unaffected; §10 trains on both and reports the comparison.

    Frequency is cumulative from the start of whatever is passed in: both variants
    divide a running blink count by the time since index 0 of `vdf`, which is Eq. 5 as
    the released code writes it. On a full recording that transient touches only the
    first sequence, but a clip cut from the middle restarts the counter, so its opening
    blinks carry a frequency that depends on when the first blink arrives. This is a
    property of the published definition and is named in §12's limitations.
    """
    closure_runs_fn = closure_runs_fn or closure_runs_hysteresis
    ear = vdf["ear_cal"].values.astype(np.float64)
    n_frames = len(ear)
    if n_frames < 10:
        return pd.DataFrame()
    sm = medfilt(ear, kernel_size=BLINK_MEDFILT)
    pad = max(1, int(round(BLINK_PAD_S * eff_fps)))

    rows, n_so_far = [], 0
    for c0, c1 in closure_runs_fn(ear):
        lo, hi = max(0, c0 - pad), min(n_frames, c1 + pad)
        seg = sm[lo:hi]
        for (s_i, b_i, e_i) in blink_retrieval(seg):
            start, bottom, end = lo + s_i, lo + b_i, lo + e_i
            dur_f = end - start + 1                                    # Eq. 2
            amp = (ear[start] - 2.0 * ear[bottom] + ear[end]) / 2.0    # Eq. 3
            denom = max(end - bottom, 1)
            vel_f = (ear[end] - ear[bottom]) / denom                   # Eq. 4
            n_so_far += 1
            frames_to_here = max(end + 1, 1)
            rows.append(dict(
                start_f=int(start), bottom_f=int(bottom), end_f=int(end),
                start_s=float(start / eff_fps),
                # --- published (frame-rate dependent) ---
                duration_pub=float(dur_f),
                velocity_pub=float(vel_f),
                frequency_pub=float(100.0 * n_so_far / frames_to_here),   # Eq. 5
                # --- corrected (physical units) ---
                duration_cor=float(dur_f / eff_fps),
                velocity_cor=float(vel_f * eff_fps),
                frequency_cor=float(60.0 * n_so_far / (frames_to_here / eff_fps)),
                # --- shared ---
                amplitude=float(amp)))
    return pd.DataFrame(rows)

## 7 · Blinks to sequences

In [ ]:
def unroll_blinks(blinks, feats, T=BLINK_SEQ_LEN, stride=BLINK_SEQ_STRIDE):
    """Slide a T-blink window over each video -> (X [N,T,4], meta).

    Videos with fewer than T blinks produce one left-zero-padded sequence, matching
    unroll_in_time in the reference implementation. After stage B a zero sits at the
    subject's own alert mean, so padded steps take a neutral value rather than an
    artefact, and stage C shifts that constant for every step alike. This is why zero
    padding is acceptable here, and why §12 refuses a clip too short to fill a
    sequence: a mostly-padded sequence is a read-out of the padding vector.
    """
    seqs, meta = [], []
    for vid, g in blinks.groupby("video_id"):
        g = g.sort_values("start_f")
        V = g[feats].values.astype(np.float32)
        n = len(V)
        if n == 0:
            continue
        base = dict(video_id=vid, subject=g.subject.iloc[0], state=int(g.state.iloc[0]),
                    split=g.split.iloc[0])
        if n <= T:
            pad = np.zeros((T, len(feats)), dtype=np.float32)
            pad[-n:, :] = V
            seqs.append(pad); meta.append(dict(base, seq_i=0, n_blinks=n, padded=True))
        else:
            for k, s0 in enumerate(range(0, n - T + 1, stride)):
                seqs.append(V[s0:s0 + T])
                meta.append(dict(base, seq_i=k, n_blinks=T, padded=False))
    return np.stack(seqs), pd.DataFrame(meta)

## 8 · Model

Feature-transformation FC, then the recurrent core, then a `10 * sigmoid` head. The
architecture is read from the checkpoint, so this class only has to match it.

In [ ]:
class AttnPool(nn.Module):
    """Additive attention over time - multiple-instance learning over time steps
    (Ilse et al. 2018). The weights say which moments the model treated as
    evidence, which is what the demo timeline draws."""
    def __init__(self, dim, hidden=64):
        super().__init__()
        self.proj = nn.Linear(dim, hidden); self.score = nn.Linear(hidden, 1)
    def forward(self, h):
        a = self.score(torch.tanh(self.proj(h))).squeeze(-1)
        w = torch.softmax(a, dim=1)
        return (h * w.unsqueeze(-1)).sum(1), w

class BlinkRegressor(nn.Module):
    """Feature-transformation FC -> recurrent core -> FC stack -> 10*sigmoid.

    `core` selects the temporal model, the only thing that varies across the §10
    comparison:
      'lstm'    stacked unidirectional LSTM - the paper's own ablation (61.4%)
      'attn'    attention-pooled BiLSTM, carried over from the frame-window work
      'fc'      no temporal model at all - the paper's second ablation (57.0%)
    HM-LSTM is not implemented: it is a custom TensorFlow-1 cell, and the paper's own
    ablation puts it 3.8 points above the plain LSTM, which is smaller than the seed
    spread here. Comparisons are therefore made against the plain LSTM figure rather
    than against the 65.2% headline.
    """
    def __init__(self, in_dim=4, pre_fc=32, hidden=32, layers=4, core="lstm", dropout=0.0):
        super().__init__()
        self.core_kind = core
        self.pre = nn.Sequential(nn.Linear(in_dim, pre_fc), nn.BatchNorm1d(pre_fc), nn.ReLU())
        if core == "fc":
            D = pre_fc * BLINK_SEQ_LEN
        elif core == "attn":
            self.rnn = nn.LSTM(pre_fc, hidden, 1, batch_first=True, bidirectional=True)
            self.pool = AttnPool(hidden * 2)
            D = hidden * 2
        else:
            self.rnn = nn.LSTM(pre_fc, hidden, layers, batch_first=True,
                               dropout=dropout if layers > 1 else 0.0)
            D = hidden
        self.head = nn.Sequential(
            nn.Linear(D, 16), nn.BatchNorm1d(16), nn.ReLU(),
            nn.Linear(16, 16), nn.BatchNorm1d(16), nn.ReLU(),
            nn.Linear(16, 8), nn.BatchNorm1d(8), nn.ReLU(),
            nn.Linear(8, 1))

    def forward(self, x, return_attn=False):
        B, T, _ = x.shape
        h = self.pre(x.reshape(B * T, -1)).reshape(B, T, -1)
        w = None
        if self.core_kind == "fc":
            z = h.reshape(B, -1)
        elif self.core_kind == "attn":
            o, _ = self.rnn(h)
            z, w = self.pool(o)
        else:
            o, _ = self.rnn(h)
            z = o[:, -1, :]
        out = 10.0 * torch.sigmoid(self.head(z))
        return (out, w) if return_attn else out

@torch.no_grad()
def predict_scores(model, X, batch=256):
    model.eval(); out = []
    for i in range(0, len(X), batch):
        xb = torch.from_numpy(X[i:i + batch]).to(DEVICE)
        out.append(model(xb).cpu().numpy().ravel())
    return np.clip(np.concatenate(out), 0, 10)

def discretise(scores):
    """Eq. 10 via the code's integer-division form: index = score // 3.34."""
    return LABELS_POOL[np.clip((np.asarray(scores) // DISCRETISE_STEP).astype(int), 0, 2)]

## 9 · Scoring a clip

`analyse_clip_blinks` is the whole inference path and carries the two guards from §12:
a hard minimum blink count, so a clip too short to fill a sequence is refused rather than
padded, and a train/serve input-range check that labels an output computed under a
calibration that does not match training.

In [ ]:
# =============================================================================
# 12.  demo - score a clip through the blink pipeline
# =============================================================================
# The inference path must be the same pipeline as training. Applying only stage C to a
# model trained with stage B is the easy version of that mistake, and it is guarded
# against below.
#
# Two limitations, stated up front.
#
# 1. calibration. Per-subject calibration needs an alert reference from the same person.
#    Without one, analyse_clip_blinks falls back to population statistics (§11), which is
#    not the protocol the model was trained under: it removes population-level scale but
#    not individual blink morphology, and §10.1 D3 shows those differences survive
#    calibration. Pass ref_clip where an enrolment recording exists. The mode is returned
#    and printed, never hidden.
#
# 2. clip length. The input unit is 30 blink events, 2-3 minutes of video. A 30 s clip
#    yields ~7 blinks and unroll_blinks left-zero-pads the rest; after stage B a zero means
#    "this subject's alert mean", so most of the sequence votes alert. That is what happened
#    here: every clip returned alert, and blink count and score correlated at r = -0.66.
#    Two guards now prevent it recurring - a hard minimum blink count, and a train/serve
#    input-range check.
#
# Both are properties of the published method, so they are written up as deployment
# limitations rather than engineered around with padding.
from collections import namedtuple

ClipVerdict = namedtuple("ClipVerdict",
                         "ok reason df blinks seq_df eff_fps calib_mode")

def _refuse(reason):
    return ClipVerdict(False, reason, None, None, None, None, None)

def _physiology_snapshot(blinks, duration_s):
    """Physiological quantities used in §7.5/§6.2, not the frame-based model channels.

    The dissertation validates blink rate per minute, median blink duration in seconds,
    median eye-opening velocity in calibrated EAR per second, and reports amplitude
    descriptively. Keeping these definitions here prevents the dashboard from labelling
    frequency_pub or duration_pub as if they were the physiological measurements.
    """
    if blinks is None or len(blinks) == 0 or duration_s is None or duration_s <= 0:
        return None
    return {
        "rate_pm": float(len(blinks) / (duration_s / 60.0)),
        "duration_s": float(blinks["duration_cor"].median()),
        "velocity": float(blinks["velocity_cor"].median()),
        "amplitude": float(blinks["amplitude"].median()),
    }

def _physiology_table(blinks, duration_s, baseline=None, current_label="clip so far"):
    """Presentation table using the same physiological definitions as the dissertation.

    If a usable alert-enrolment baseline exists, raw changes are shown relative to that
    driver. Without enrolment there is deliberately no invented 'personal baseline':
    the model can still use its population Stage-B fallback, but that is a model
    calibration statistic, not a physiological alert measurement for this driver.
    """
    cur = _physiology_snapshot(blinks, duration_s)
    specs = [
        ("blink rate", "rate_pm", "/min", "↑ higher (P1)"),
        ("blink duration", "duration_s", "s", "↑ longer (P2)"),
        ("eye-opening velocity", "velocity", "EAR/s", "↓ slower (P3)"),
        ("blink amplitude", "amplitude", "EAR", "descriptive only"),
    ]

    def fmt(k, v, unit):
        if v is None or not np.isfinite(v):
            return "—"
        if k == "rate_pm":
            return f"{v:.1f} {unit}"
        return f"{v:.3f} {unit}"

    rows = []
    for label, key, unit, expected in specs:
        now = None if cur is None else cur.get(key)
        base = None if baseline is None else baseline.get(key)
        if base is None or now is None or not np.isfinite(base) or abs(base) < 1e-12:
            change = "—"
        else:
            change = f"{(now / base - 1.0) * 100:+.1f}%"
        rows.append({
            "physiological measure": label,
            "alert enrolment": fmt(key, base, unit),
            current_label: fmt(key, now, unit),
            "change vs alert": change,
            "expected with fatigue": expected,
        })
    return pd.DataFrame(rows)

def load_deliverable(path=None):
    path = path or CKPT_PATH
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    if ck.get("feature_variant") not in ("published", "corrected"):
        raise RuntimeError("checkpoint predates the blink pipeline - retrain (§9).")
    if ck.get("seq_len") != BLINK_SEQ_LEN:
        raise RuntimeError(f"sequence length mismatch: checkpoint {ck.get('seq_len')} "
                           f"vs notebook {BLINK_SEQ_LEN}.")
    m = BlinkRegressor(in_dim=4, pre_fc=ck["arch"]["pre_fc"], hidden=ck["arch"]["hidden"],
                       layers=ck["arch"]["layers"], core=ck["arch"]["core"]).to(DEVICE)
    m.load_state_dict(ck["state_dict"]); m.eval()
    return m, ck

def _assert_helpers_intact():
    """Fail fast if a scratch variable has rebound a module-level helper.

    A rebinding like `_d = np.abs(np.diff(x))` would surface as "'numpy.ndarray' object
    is not callable" inside MediaPipe frame processing, far from its cause. The helpers
    are renamed so this cannot happen by accident; this check catches any recurrence in
    under a millisecond.
    """
    for _n in ("_lm_dist", "_lm_ear", "_lm_mar", "prepare_video",
               "extract_blinks_for_video", "unroll_blinks", "discretise"):
        _o = globals().get(_n)
        if not callable(_o):
            raise RuntimeError(
                f"`{_n}` is no longer callable - it is now {type(_o).__name__}. Some "
                f"cell has rebound it as a variable. Re-run the defining cell, find "
                f"the assignment, and rename that variable.")

def analyse_clip_blinks(video_path, model, ck, max_frames=20000, ref_clip=None,
                        allow_padding=False):
    """-> ClipVerdict(ok, reason, df, blinks, seq_df, eff_fps, calib_mode).

    Runs the full training pipeline on one clip: geometry -> blinks -> features ->
    calibration -> global z-score -> model. Refuses rather than guessing when the clip
    cannot support a real sequence; `ok=False` always carries a `reason` to display.
    """
    _assert_helpers_intact()
    FE = FEATS_PUB if ck["feature_variant"] == "published" else FEATS_COR

    # Enrolment, when supplied, drives both calibration stages.
    ref_blinks, ear_ref, phys_baseline = None, None, None
    if ref_clip is not None:
        rraw = extract_video(ref_clip, ck["sample_every"], max_frames, progress=False)
        rdf = prepare_video(rraw.assign(video_id="ref", subject="demo", state=0, split="demo"))
        ear_ref = float(rdf.attrs["ear_ref"])
        ref_eff = float(rdf.eff_fps.iloc[0])
        ref_blinks = extract_blinks_for_video(rdf, ref_eff)
        ref_dur = len(rdf) / ref_eff
        # Four blinks is the minimum already used for Stage-B calibration. Below that,
        # do not present a noisy physiology baseline as if it were a reliable alert reference.
        if len(ref_blinks) >= 4:
            phys_baseline = _physiology_snapshot(ref_blinks, ref_dur)

    raw = extract_video(video_path, ck["sample_every"], max_frames, progress=True)
    raw["video_id"] = "demo"; raw["subject"] = "demo"; raw["state"] = -1; raw["split"] = "demo"
    df = prepare_video(raw, ear_ref=ear_ref)
    eff = float(df.eff_fps.iloc[0])
    b = extract_blinks_for_video(df, eff)
    T = int(ck["seq_len"])
    if len(b) == 0:
        return _refuse("No blinks retrieved - the face may not be tracked, or the eyes "
                       "may not be visible.")
    if len(b) < T and not allow_padding:
        _mins = len(df) / eff / 60.0
        return _refuse(
            f"Only {len(b)} blinks retrieved from {_mins:.1f} min of footage; the model's "
            f"input unit is {T} blink events (Ghoddoosian et al. 2019). Zero-padding to "
            f"{T} would make {(T - len(b)) / T:.0%} of the sequence padding, and after "
            f"per-subject calibration a padded step sits at the alert mean, so the "
            f"verdict would largely be reading its own padding. Refusing rather than "
            f"answering. Supply ~{max(2.0, T / max(len(b) / max(_mins, 1e-6), 1e-6)):.1f} "
            f"min of footage at this blink rate.")

    if ref_blinks is not None and len(ref_blinks) >= 4:
        mu = ref_blinks[FE].mean()
        sd = ref_blinks[FE].std().replace(0.0, 1e-6)
        calib_mode = (f"enrolment clip, {len(ref_blinks)} reference blinks "
                      f"(personalised Stage-A and Stage-B calibration) "
                      f"| stage-A open-eye ref {ear_ref:.4f}")
    elif ref_clip is not None:
        mu = pd.Series(ck["fallback_mu"], index=FE)
        sd = pd.Series(ck["fallback_sd"], index=FE)
        calib_mode = (f"population fallback for stage B (reference clip yielded only "
                      f"{0 if ref_blinks is None else len(ref_blinks)} blinks - need >=4; "
                      f"stage A still personalised, stage-A open-eye ref {ear_ref:.4f})")
    else:
        mu = pd.Series(ck["fallback_mu"], index=FE)
        sd = pd.Series(ck["fallback_sd"], index=FE)
        calib_mode = ("population fallback (no per-driver reference - out of protocol; "
                      "Stage A uses this clip's own P90)")

    # Preserve presentation-only context without changing the ClipVerdict API.
    df.attrs["phys_baseline"] = phys_baseline
    df.attrs["stage_a_ref"] = float(df.attrs.get("ear_ref", np.nan))

    bc = b.copy()
    bc[FE] = (b[FE] - mu) / sd
    bc["video_id"] = "demo"; bc["subject"] = "demo"; bc["state"] = -1; bc["split"] = "demo"
    X, meta = unroll_blinks(bc, FE)
    X = ((X - np.array(ck["global_mu"])) / np.array(ck["global_sd"])).astype(np.float32)

    # Train/serve skew is displayed as a warning rather than silently changing the result.
    _oob = float(np.mean(np.abs(X) > 5.0))
    if _oob > 0.01:
        calib_mode += (f" | WARNING: {_oob:.1%} of normalised inputs exceed 5 sigma "
                       f"(max |z| = {np.abs(X).max():.1f}). Treat this verdict as unreliable.")

    scores = predict_scores(model, X)
    seq_df = pd.DataFrame(dict(seq_i=meta.seq_i.values, score=scores,
                               band=discretise(scores)))
    starts = bc.sort_values("start_f").start_s.values
    seq_df["t_start_s"] = [float(starts[min(len(starts) - 1, i * BLINK_SEQ_STRIDE)])
                           for i in seq_df.seq_i]
    seq_df["t_end_s"] = [float(starts[min(len(starts) - 1,
                          i * BLINK_SEQ_STRIDE + BLINK_SEQ_LEN - 1)]) for i in seq_df.seq_i]
    return ClipVerdict(True, None, df, b, seq_df, eff, calib_mode)

def clip_vote(seq_df):
    """Three-class clip verdict: majority over sequence bands, mean-score tie-break.

    This is the same rule used by the live path. Keeping it in one helper prevents the
    Summary and Rendered-overlay paths from silently using pandas idxmax() on ties.
    """
    if seq_df is None or len(seq_df) == 0:
        return None
    counts = seq_df["band"].value_counts()
    mx = int(counts.max())
    winners = [float(k) for k, v in counts.items() if int(v) == mx]
    if len(winners) == 1:
        return winners[0]
    return float(discretise([float(seq_df["score"].mean())])[0])

def blink_timeline_figure(df, b, seq_df, eff, title=""):
    """EAR trace with retrieved blinks marked, and the regression score over time."""
    t = np.arange(len(df)) / eff
    fig, ax = plt.subplots(2, 1, figsize=(12, 5.2), sharex=True,
                           gridspec_kw=dict(height_ratios=[1, 1]))
    ax[0].plot(t, df.ear_cal, lw=0.9, color="#2F6F5B")
    for _, r in b.iterrows():
        ax[0].axvspan(r.start_f / eff, r.end_f / eff, color="#B33A31", alpha=0.22)
        ax[0].plot(r.bottom_f / eff, df.ear_cal.iloc[int(r.bottom_f)], "v", ms=4, color="#B33A31")
    ax[0].set_ylabel("EAR (calibrated)")
    ax[0].set_title(f"{title} - {len(b)} blinks retrieved")

    ax[1].step(seq_df.t_end_s, seq_df.score, where="post", lw=1.6, color="#1A1D24")
    ax[1].axhline(CUT_LO, color="#888", ls="--", lw=1)
    ax[1].axhline(CUT_HI, color="#888", ls="--", lw=1)
    ax[1].fill_between(seq_df.t_end_s, 0, CUT_LO, color="#9FE1CB", alpha=.25)
    ax[1].fill_between(seq_df.t_end_s, CUT_LO, CUT_HI, color="#E8D9A8", alpha=.28)
    ax[1].fill_between(seq_df.t_end_s, CUT_HI, 10, color="#F5C4B3", alpha=.35)
    ax[1].set_ylim(0, 10); ax[1].set_ylabel("drowsiness score")
    ax[1].set_xlabel("time (s)  ·  each step = one 30-blink sequence")
    fig.tight_layout()
    return fig

In [ ]:
# ---- optional smoke test --------------------------------------------------------
# Loading the checkpoint is cheap; scoring a full 2-3 minute clip is not. The old notebook
# did that before the dashboard could even launch, which made Run all look stuck.
RUN_SMOKE_TEST = False

_model, _ck = load_deliverable()
print(f"checkpoint loaded: {_ck['arch']['core']} core, {_ck['feature_variant']} features, "
      f"seq_len {_ck['seq_len']}")

if RUN_SMOKE_TEST:
    _test = _demo[0]
    _subj = re.match(r"s(\d+)_", _test.stem)
    _ref = (next(iter(sorted(DEMO_DIR.glob(f"s{_subj.group(1)}_enrol_*.mp4"))), None)
            if _subj else None)
    _res = analyse_clip_blinks(str(_test), _model, _ck,
                               ref_clip=str(_ref) if _ref else None)
    if _res.ok:
        print(f"{_test.stem}: {len(_res.blinks)} blinks, {len(_res.seq_df)} sequences, "
              f"mean score {_res.seq_df.score.mean():.1f}/10, vote {clip_vote(_res.seq_df):.0f}")
        print(f"calibration: {_res.calib_mode}")
    else:
        print(f"{_test.stem}: not scored - {_res.reason}")
else:
    print("video smoke test skipped (set RUN_SMOKE_TEST = True if you want it)")


checkpoint loaded: lstm core, published features, seq_len 30
video smoke test skipped (set RUN_SMOKE_TEST = True if you want it)


## 10 · Annotated overlay

Renders the mesh, the eye and mouth contours, live EAR/MAR, the four features of the most
recent retrieved blink, and the running band. Slow, so the dashboard serves the
pre-rendered file from §12.1 where one exists.

In [ ]:
# CRF 20 rather than 28: 28 did not keep the 1-pixel landmark mesh legible after H.264.
# These files live on Drive and are never embedded at full length.
OVERLAY_CRF = 20

def _hud_panel(img, x0, y0, x1, y1, colour=(20, 20, 24), alpha=0.55):
    # translucent rounded-off rectangle for HUD text (simple alpha blend on the ROI)
    x0, y0 = max(int(x0), 0), max(int(y0), 0)
    x1, y1 = min(int(x1), img.shape[1]), min(int(y1), img.shape[0])
    if x1 <= x0 or y1 <= y0:
        return
    roi = img[y0:y1, x0:x1]
    over = np.empty_like(roi); over[:] = colour
    img[y0:y1, x0:x1] = cv2.addWeighted(roi, 1 - alpha, over, alpha, 0)

def _hud_text_size(txt, fs, th=1):
    (w, h), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, fs, th)
    return w, h

def _hud_block(img, anchor_x, y0, rows, pad=11, gap=9, colour=(18, 18, 22),
               alpha=0.58, align="left"):
    """Draw a translucent panel sized from its own text, then the text on it.

    `rows` is a list of (text, bgr, font_scale, thickness); None inserts a hairline
    separator. Returns the panel rectangle, so a caller can hang a bar off the panel it
    belongs to instead of guessing a pixel offset.

    The panel is measured with `cv2.getTextSize` at the renderer's own font and scale
    rather than sized as a fraction of frame width, which does not survive real strings
    at real frame sizes and left text and bars overhanging their backdrops.
    """
    sizes = [_hud_text_size(r[0], r[2], r[3]) if r else (0, 0) for r in rows]
    w = max([s[0] for s in sizes] + [1])
    x0 = anchor_x if align == "left" else anchor_x - w - 2 * pad
    y, ys = y0 + pad, []
    for i, r in enumerate(rows):
        if r is None:
            y += gap + 3; ys.append(y - gap // 2 - 2)
        else:
            y += sizes[i][1]; ys.append(y); y += gap
    x1, y1 = x0 + w + 2 * pad, y - gap + pad
    _hud_panel(img, x0, y0, x1, y1, colour=colour, alpha=alpha)
    for i, r in enumerate(rows):
        if r is None:
            cv2.line(img, (int(x0 + pad), int(ys[i])), (int(x1 - pad), int(ys[i])),
                     (105, 105, 112), 1)
        else:
            cv2.putText(img, r[0], (int(x0 + pad), int(ys[i])),
                        cv2.FONT_HERSHEY_SIMPLEX, r[2], r[1], r[3], cv2.LINE_AA)
    return int(x0), int(y0), int(x1), int(y1)

In [ ]:
def render_annotated_video(video_path, model, ck, out_path, max_s=None,
                           true_label=None, target_w=720, ref_clip=None):
    """Analyse a clip with the trained model, then re-read the source at full fps and draw:
    scaled landmark mesh + eye/mouth contours, live EAR/MAR/pitch, rolling P(drowsy) bar,
    alert/drowsy verdict at the safety threshold, true label (if given) and event banners.

    The analysed span and the rendered span are identical by construction. If the analysis
    consumed the whole clip while only max_s seconds were rendered, the verdict could be
    driven by events the viewer never saw. max_s=None renders everything.

    Returns a dict with the metrics and the raw analysis (so callers can plot the timeline
    without paying for a second MediaPipe pass), or None if the clip is too short."""
    se = ck["sample_every"]
    _cap = cv2.VideoCapture(str(video_path))
    fps = _cap.get(cv2.CAP_PROP_FPS) or 30.0
    n_total = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    _cap.release()
    n_src = n_total if max_s is None else min(n_total, int(max_s * fps))
    if n_src <= 0:
        n_src = int((max_s or 30) * fps)

    # The overlay shows the regression score of the sequence ending at or before each frame and
    # marks retrieved blinks. ref_clip threads the subject's enrolment clip through, so an
    # overlay matches the in-protocol column of §12.1 rather than silently using the fallback.
    res = analyse_clip_blinks(video_path, model, ck, ref_clip=ref_clip,
                              max_frames=int(np.ceil(n_src / se)))
    if not res.ok:
        print(f"  cannot render {Path(video_path).name}: {res.reason}")
        return None
    df, blk, seq_df, eff = res.df, res.blinks, res.seq_df, res.eff_fps
    # ---- per-frame score, and the warm-up gate ------------------------------
    # Warm-up gate. The input unit is 30 blink events, so on a three-minute clip the first
    # sequence typically completes 60-75% of the way through. Filling backwards with
    # np.interp(..., left=) would display a verdict derived from blinks that had not happened
    # yet, beside a blink counter reading a handful. Frames before the first sequence now show
    # no verdict; the score is still interpolated between completed sequences.
    _t = np.arange(len(df)) / eff
    _score = np.interp(_t, seq_df.t_end_s.values, seq_df.score.values,
                       left=float(seq_df.score.iloc[0]), right=float(seq_df.score.iloc[-1]))
    _ready_from_s = float(seq_df.t_end_s.iloc[0])     # first completed sequence
    win_df = pd.DataFrame(dict(start_s=seq_df.t_end_s, end_s=seq_df.t_end_s,
                               p_drowsy=seq_df.score / 10.0))
    heat = _score / 10.0

    # ---- the clip-level verdict, by the same rule §12.1 prints ---------------
    # §12.1 takes the majority over sequence bands, and this reproduces that rule so the overlay
    # and the table beside it cannot disagree about the clip-level answer. Both this and the
    # running band are shown and labelled, because they legitimately differ.
    _BANDN = {0.0: "ALERT", 5.0: "LOW VIGILANCE", 10.0: "DROWSY"}
    _BANDC = {0.0: (90, 200, 90), 5.0: (40, 170, 235), 10.0: (60, 60, 230)}
    _clip_band = float(clip_vote(seq_df))
    _clip_verdict = _BANDN[_clip_band]
    ev = pd.DataFrame([dict(type="blink", start_s=float(r.start_f / eff),
                            duration_s=float((r.end_f - r.start_f) / eff))
                       for _, r in blk.iterrows()])

    # The HUD shows the four published blink features the model consumes, not just the raw
    # EAR/MAR traces, so the chain is visible: geometry -> retrieved blink -> features -> score.
    # Values are those of the most recent blink, plus a rolling blink rate over 60 s.
    _dur_col = "duration_pub" if "duration_pub" in blk.columns else "duration_s"
    _vel_col = "velocity_pub" if "velocity_pub" in blk.columns else "velocity"
    _blk_end = (blk.end_f.values / eff) if len(blk) else np.array([])
    def _last_blink_at(t_s):
        if not len(_blk_end):
            return None
        _j = int(np.searchsorted(_blk_end, t_s, side="right")) - 1
        return blk.iloc[_j] if _j >= 0 else None
    def _rate_at(t_s, win=60.0):
        if not len(_blk_end):
            return 0.0
        _n = int(((_blk_end > max(t_s - win, 0)) & (_blk_end <= t_s)).sum())
        return _n * 60.0 / min(win, max(t_s, 1.0))
    thr = float(ck.get("cut_hi", 6.6)) / 10.0
    rot = detect_rotation(video_path)
    lm  = new_landmarker()                      # fresh main-thread landmarker

    # sampled-frame -> active event type (blinks excluded: they'd flash constantly)
    ev_type = [None] * len(df)
    for _, e in ev.iterrows():
        if e.type == "blink":
            continue
        a = int(e.start_s * eff); b = int((e.start_s + e.duration_s) * eff)
        for i in range(max(a, 0), min(b + 1, len(df))):
            ev_type[i] = e.type

    cap = cv2.VideoCapture(str(video_path))
    ok, first = cap.read()
    if not ok:
        cap.release(); return None
    if rot is not None:
        first = cv2.rotate(first, rot)
    H0, W0 = first.shape[:2]
    scale = min(1.0, target_w / W0)
    W = int(W0 * scale) // 2 * 2; H = int(H0 * scale) // 2 * 2   # even dims for H.264
    n_out = int(min(n_src, len(df) * se))     # never render past what the model analysed
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    # Frames go straight into ffmpeg over a pipe: one lossy generation, no temp file. The
    # earlier mp4v-intermediate path cost 2.3 dB PSNR overall and 2.35 dB on luma, which is
    # where the mesh dots and HUD text live.
    # Note §12.1 skips any overlay whose output file exists, so delete ANNOT_DIR to re-render.
    _enc = subprocess.Popen(
        ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "bgr24",
         "-s", f"{W}x{H}", "-r", f"{fps:.6f}", "-i", "pipe:0",
         "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "veryfast",
         "-crf", str(OVERLAY_CRF), "-movflags", "+faststart", str(out_path)],
        stdin=subprocess.PIPE)
    fs = float(np.clip(W / 1100, 0.45, 0.70))   # font scale tied to output width
    u  = fs / 0.6                               # HUD unit for pixel offsets

    for fi in tqdm(range(n_out), desc=f"  overlay {Path(video_path).stem}",
                   unit="frame", leave=False):
        ok, frame = cap.read()
        if not ok:
            break
        if rot is not None:
            frame = cv2.rotate(frame, rot)
        if scale < 1.0:
            frame = cv2.resize(frame, (W, H))
        si = min(len(df) - 1, fi // se)          # model-signal index for this source frame

        # ---- face overlay (auto-scaled to face width: not too huge, not too small) ----
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        det = lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        ear_now = mar_now = float("nan")
        if det.face_landmarks:
            l = det.face_landmarks[0]
            P  = lambda i: (int(l[i].x * W), int(l[i].y * H))
            Pf = lambda i: (l[i].x * W, l[i].y * H)
            xs = [p.x for p in l]
            face_w = (max(xs) - min(xs)) * W
            r_mesh = int(np.clip(face_w * 0.006, 1, 3))
            r_key  = int(np.clip(face_w * 0.011, 2, 5))
            th     = int(np.clip(face_w * 0.005, 1, 2))
            for i in range(0, 468, 3):                       # sparse mesh, subtle grey
                cv2.circle(frame, P(i), r_mesh, (185, 185, 185), -1)
            cv2.polylines(frame, [np.array([P(i) for i in LEFT_EYE])],  True, (0, 255, 0), th)
            cv2.polylines(frame, [np.array([P(i) for i in RIGHT_EYE])], True, (0, 255, 0), th)
            cv2.polylines(frame, [np.array([P(i) for i in [61, 13, 291, 14]])],
                          True, (0, 165, 255), th)
            for i in LEFT_EYE + RIGHT_EYE: cv2.circle(frame, P(i), r_key, (0, 255, 0), -1)
            for i in MOUTH:                cv2.circle(frame, P(i), r_key, (0, 165, 255), -1)
            ear_now = (_lm_ear([Pf(i) for i in LEFT_EYE]) + _lm_ear([Pf(i) for i in RIGHT_EYE])) / 2
            mar_now = _lm_mar([Pf(i) for i in MOUTH])
        else:
            cv2.putText(frame, "NO FACE", (12, int(34 * u)), cv2.FONT_HERSHEY_SIMPLEX,
                        fs * 1.2, (0, 0, 255), 2, cv2.LINE_AA)

        # ---- what the model can honestly say at this instant -----------------
        _t_now = si / eff
        _ready = _t_now >= _ready_from_s
        p_now  = float(heat[si])
        _n_now = (int(np.searchsorted(_blk_end, _t_now, side="right"))
                  if len(_blk_end) else 0)
        if _ready:
            _s10 = float(np.clip(p_now * 10.0, 0, 10))
            # take the band from discretise() rather than comparing against CUT_LO/CUT_HI: those are
            # 3.3/6.6 while discretise divides by 3.34
            _band = float(discretise([_s10])[0])
            verdict, vcol = _BANDN[_band], _BANDC[_band]
        else:
            _s10, verdict, vcol = float("nan"), "WARMING UP", (170, 170, 170)
        _lb = _last_blink_at(_t_now)

        # ---- HUD, top-left: grouped by what kind of number each value is -----
        # three labelled groups: what the camera measures now, what the detector last retrieved,
        # and what the model concluded, so the score never reads as part of the raw EAR trace
        _lbl = (150, 190, 190)
        _hud_block(frame, 8, 8, [
            ("MEASURED NOW", _lbl, fs * 0.72, 1),
            (f"EAR {ear_now:.3f}   MAR {mar_now:.3f}", (255, 255, 0), fs, 1),
            (f"pitch {df.pitch.iloc[si]:+5.0f}   ear_cal {df.ear_cal.iloc[si]:.2f}",
             (255, 255, 0), fs, 1),
            None,
            ("LAST RETRIEVED BLINK", _lbl, fs * 0.72, 1),
            (f"dur {float(_lb[_dur_col]):.2f}   amp {float(_lb.amplitude):.2f}   "
             f"vel {float(_lb[_vel_col]):.2f}" if _lb is not None
             else "none retrieved yet", (215, 235, 235), fs, 1),
            (f"count {_n_now}   rate {_rate_at(_t_now):.1f}/min", (215, 235, 235), fs, 1),
            None,
            ("MODEL", _lbl, fs * 0.72, 1),
            ((f"score {_s10:.1f}/10   ->   {verdict}") if _ready
             else f"warming up   {_n_now}/{int(ck['seq_len'])} blinks",
             vcol if _ready else (190, 190, 190), fs, 1),
        ])

        # ---- HUD, top-right: both verdicts, each labelled --------------------
        _rows_r = [("NOW", (150, 150, 150), fs * 0.68, 1),
                   (verdict, vcol, fs * 1.0, 2),
                   None,
                   (f"CLIP VERDICT  (majority of {len(seq_df)})",
                    (150, 150, 150), fs * 0.62, 1),
                   (_clip_verdict, _BANDC[_clip_band], fs * 0.85, 2)]
        if true_label:
            _rows_r.append((f"true: {true_label}", (205, 205, 205), fs * 0.7, 1))
        _rx0, _ry0, _rx1, _ry1 = _hud_block(frame, W - 8, 8, _rows_r,
                                            colour=(15, 15, 15), align="right")
        # the bar hangs off the panel it belongs to, not off a guessed offset
        _bh, _by = max(6, int(8 * u)), _ry1 + int(5 * u)
        cv2.rectangle(frame, (_rx0, _by), (_rx1, _by + _bh), (70, 70, 70), -1)
        if _ready:
            _fill = int(_rx0 + (_rx1 - _rx0) * min(max(p_now, 0.0), 1.0))
            cv2.rectangle(frame, (_rx0, _by), (_fill, _by + _bh),
                          (0, int(200 * (1 - p_now)), int(230 * p_now)), -1)

        # ---- event banner (bottom-centre) while active ----
        et = ev_type[si]
        if et:
            txt = et.upper()
            (tw, thh), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, fs * 1.2, 2)
            x0 = (W - tw) // 2
            _hud_panel(frame, x0 - 14, H - thh - 36, x0 + tw + 14, H - 10,
                       colour=(30, 30, 160) if et == "microsleep" else (20, 90, 160), alpha=0.7)
            cv2.putText(frame, txt, (x0, H - 20), cv2.FONT_HERSHEY_SIMPLEX,
                        fs * 1.2, (255, 255, 255), 2, cv2.LINE_AA)
        _enc.stdin.write(np.ascontiguousarray(frame).tobytes())

    _enc.stdin.close(); _enc.wait(); cap.release()

    mean_p = float(win_df.p_drowsy.mean())
    return dict(out_path=Path(out_path), mean_p=mean_p,
                clip_verdict=_clip_verdict,          # three-class, by majority - what is drawn
                # named explicitly: this is a two-class threshold on the clip mean, not the three-class
                # majority the overlay draws
                binary_verdict="DROWSY" if mean_p >= thr else "ALERT",
                n_events=(dict(ev.type.value_counts()) if len(ev) else {}),
                duration_s=n_out / fps, n_windows=len(win_df),
                analysis=res)      # (df, win_df, heat, ev, eff) for the caller's timeline plot

## 11 · Dashboard

Three tabs over one clip.

**Live** is the one to demonstrate. It reads the clip frame by frame and scores it as the
evidence arrives — the EAR trace extends, blinks are counted, and the verdict appears only
once 30 blink events exist. Nothing is precomputed, so what you watch is what a car would
experience, including the minutes of silence at the start. It requires an enrolment clip,
because without one the Stage-A reference is the clip's own 90th percentile and is not
knowable until the clip ends. When the clip finishes it fills in the other two tabs from
the same run.

**Summary** is the batch path: score everything, then report. It works without enrolment
and labels the result out of protocol when it is.

**Rendered overlay** plays the annotated video — the one a live run just wrote, or 12.1's
pre-rendered file where one exists.

The clip and calibration controls on the left apply to all three. `up` accepts a webcam
recording, so you can record yourself and score it against your own alert reference.

### The knobs, in the live-scoring cell

| Setting | Default | Effect |
|---|---|---|
| `LIVE_TRANSPORT` | `"mjpeg"` | frames leave over one multipart connection. Set `"inline"` if a tunnel will not carry it: fewer frames a second, no extra route |
| `LIVE_DETECT_W` | `None` | full-frame detection, matching the batch pipeline. The agreement check below measures what `480` would cost in EAR and save in time |
| `LIVE_PUBLISH_HZ` | `20` | frames a second on the wire. Lower it if the link is the constraint |
| `LIVE_ANALYSE_S` | `2.0` | seconds of footage between analysis passes. Each pass re-derives the blinks from the whole buffer, so the cost grows with clip length |
| `LIVE_WRITE_OVERLAY` | `True` | pipe the drawn frames to ffmpeg, so the overlay tab is ready when the clip ends |

The final markdown line of a live run reports frames processed, wall clock, the real-time
factor, and milliseconds per frame for landmarking and for drawing — enough to say which
of the three is the constraint on the machine it is running on.

In [ ]:
# =============================================================================
# 12.3  dashboard
# =============================================================================
# Four questions, in order, and nothing else:
#   1. What is the verdict, and how confident is the vote?
#   2. Was the driver enrolled - the protocol the model was trained under, or the weaker
#      population fallback? This changes the verdict, so it must be visible.
#   3. What did the model measure? The four blink features, against the alert reference.
#   4. Where in the clip did the score move? One timeline.
# Removed: the raw per-blink dataframe and the duplicate summary text. The annotated video
# is served from §12.1's pre-rendered file where one exists.
import gradio as gr, tempfile

_g_model, _g_ck = load_deliverable()
_DEMO_DIR  = DEMO_DIR if "DEMO_DIR" in globals() else (MODELS_DIR.parent / "demo_clips")
_ANNOT_DIR = _DEMO_DIR / "annotated"
_BAND  = {0.0: "ALERT", 5.0: "LOW VIGILANCE", 10.0: "DROWSY"}
_FEATS = FEATS_PUB if _g_ck["feature_variant"] == "published" else FEATS_COR

def _scan(pattern, exclude=()):
    if not _DEMO_DIR.exists():
        return {}
    return {p.stem: str(p) for p in sorted(_DEMO_DIR.glob(pattern))
            if not any(x in p.stem for x in exclude)}

_clip_choices  = _scan("s*_*.mp4", exclude=("_enrol_", "annotated_"))
_bad_choices   = _scan("invalid_*.mp4")
_all_choices   = {**_clip_choices, **{f"[refusal demo] {k}": v for k, v in _bad_choices.items()}}
if not _all_choices:
    print("note: no demo clips found - dashboard runs in upload-only mode. Run §12.1 first.")

def _true_label(stem):
    return next((s for s in ("alert", "lowvig", "drowsy") if s in stem), None)

def _subject_of_stem(stem):
    m = re.match(r"s(\d+)_", stem)
    return m.group(1) if m else None

def _auto_enrolment(path):
    """The subject's own enrolment clip from §12.1, if one was cut."""
    subj = _subject_of_stem(Path(path).stem)
    if not subj or not _DEMO_DIR.exists():
        return None
    hits = sorted(_DEMO_DIR.glob(f"s{subj}_enrol_*.mp4"))
    return str(hits[0]) if hits else None

def g_analyse(choice, upload, ref_upload, use_enrolment):
    path = upload or _all_choices.get(choice)
    if not path:
        return "Pick a clip or upload a video.", None, pd.DataFrame()

    ref = ref_upload or (_auto_enrolment(path) if use_enrolment else None)
    res = analyse_clip_blinks(path, _g_model, _g_ck, ref_clip=ref)

    if not res.ok:
        return (f"## Not scored\n\n**{res.reason}**\n\n"
                "Refusing is deliberate, not a failure. The published method consumes "
                "**30 blink events**. A shorter clip would be mostly padding rather than "
                "evidence from the driver."), None, pd.DataFrame()

    df, b, seq_df, eff, calib_mode = (res.df, res.blinks, res.seq_df,
                                      res.eff_fps, res.calib_mode)
    mean_s = float(seq_df.score.mean())
    frac = seq_df.band.value_counts(normalize=True)
    band = _BAND[float(clip_vote(seq_df))]
    tl = _true_label(Path(path).stem)

    if str(calib_mode).startswith("enrolment clip,"):
        banner = ("&#9989; **In protocol** - Stage A and Stage B use this driver's "
                  "alert enrolment recording.")
    elif str(calib_mode).startswith("population fallback for stage B"):
        banner = ("&#9888;&#65039; **Out of protocol** - an alert reference was supplied, "
                  "but it produced too few blinks for Stage B. Stage A remains personalised; "
                  "Stage B uses population alert statistics.")
    else:
        banner = ("&#9888;&#65039; **Out of protocol** - no driver enrolment is available. "
                  "Stage A uses this clip's own P90 open-eye reference and Stage B uses "
                  "population alert statistics.")

    purity = float(frac.max())
    msg = (f"## {band} &nbsp;·&nbsp; {mean_s:.1f}/10\n\n"
           f"{banner}\n\n"
           f"**Vote:** " + " · ".join(f"{_BAND[float(k)].lower()} {v:.0%}"
                                      for k, v in frac.sort_index().items())
           + f" &nbsp;(purity {purity:.0%} over {len(seq_df)} sequences)\n\n"
           + (f"**True label:** {tl}\n\n" if tl else "")
           + f"<sub>{len(b)} blinks retrieved · verdict is a majority vote over "
             f"{_g_ck['seq_len']}-blink sequences with the mean as tie-break. "
             f"This dashboard demonstrates inference behaviour; measured performance is "
             f"reported from the held-out evaluation, not from selected demo clips.</sub>")

    # These are the physiological quantities validated in the dissertation:
    # rate/min, duration in seconds, opening velocity per second, and amplitude.
    # They are not mislabeled versions of the frame-based model channels.
    feat_tbl = _physiology_table(
        b, len(df) / eff,
        baseline=df.attrs.get("phys_baseline"),
        current_label="whole clip")

    fig = blink_timeline_figure(df, b, seq_df, eff, title=Path(path).stem)
    return msg, fig, feat_tbl

def g_annotated(choice, upload, ref_upload, use_enrolment):
    """Serve the overlay rendered in §12.1; render on demand only if absent."""
    path = upload or _all_choices.get(choice)
    if not path:
        return None
    ref = ref_upload or (_auto_enrolment(path) if use_enrolment else None)
    pre = _ANNOT_DIR / f"annotated_{Path(path).stem}.mp4"
    # A pre-rendered overlay is only safe to reuse for the notebook's normal automatic
    # enrolment path. A custom reference (or explicitly disabling enrolment) can change the
    # score, so render that configuration on demand instead of showing a stale verdict.
    _auto_ref = _auto_enrolment(path) if use_enrolment else None
    _pre_matches = ref_upload is None and use_enrolment and ref == _auto_ref
    if _pre_matches and pre.exists() and pre.stat().st_size > 50_000:
        return str(pre)
    out = Path(tempfile.gettempdir()) / f"annot_{Path(path).stem}.mp4"
    info = render_annotated_video(path, _g_model, _g_ck, out,
                                  true_label=_true_label(Path(path).stem),
                                  ref_clip=ref)
    return str(out) if info else None

In [ ]:
# ---- live scoring: one frame at a time -----------------------------------------
# Frames are read, landmarked and appended to a buffer. Every LIVE_ANALYSE_S seconds
# the accumulated prefix goes through the same prepare_video -> blink retrieval ->
# Stage-B calibration -> Stage-C standardisation -> model path used by Summary.
#
# With enrolment, Stage A and Stage B are fixed from the driver's alert reference.
# Without enrolment, the run is explicitly OUT OF PROTOCOL: Stage A uses the P90 of
# the footage seen so far and is recomputed over the whole prefix at each analysis
# update; Stage B uses the checkpoint's population-alert statistics. At clip end the
# running P90 is the full-clip P90, so the final live result matches Summary's
# no-enrolment condition.
#
# The physiology table uses the quantities actually validated in the dissertation:
# blink rate/min, duration in seconds, opening velocity per second, and amplitude.
import base64

LIVE_ANALYSE_S  = 2.0     # seconds of footage between analysis passes
LIVE_VIEW_W     = 460     # width sent to the browser
LIVE_DRAW_W     = 720     # width the overlay is drawn at, and written to file
LIVE_DETECT_W   = None    # width fed to MediaPipe; None = full frame. See the cell below
LIVE_SPARK_S    = 10.0    # seconds of EAR drawn on the frame
LIVE_TRANSPORT  = "inline" # portable default; use "mjpeg" only if the custom route works
LIVE_PUBLISH_HZ = 20.0    # frames per second put on the wire under "mjpeg"
LIVE_EMIT_HZ    = 5.0     # keep Gradio event traffic modest under inline transport
LIVE_JPEG_Q     = 68
LIVE_WRITE_OVERLAY = False  # avoid synchronous ffmpeg blocking the live demo; render later if needed
LIVE_REALTIME = True       # if inference is faster than the source, pace playback to source time
ENROL_CACHE     = DEMO_DIR / "enrolment_cache.json"

_SKIP = getattr(gr, "skip", gr.update)

_BANDC  = {0.0: "#2F6F5B", 5.0: "#9A7B33", 10.0: "#8C2F27"}

def _feats_of(ck):
    return FEATS_PUB if ck["feature_variant"] == "published" else FEATS_COR

_ROT_CACHE = {}

def _cached_rotation(path):
    """detect_rotation probes three timestamps in four orientations, so it costs twelve
    detect() calls. A clip's orientation does not change between runs."""
    k = str(path)
    if k not in _ROT_CACHE:
        _ROT_CACHE[k] = detect_rotation(path)
    return _ROT_CACHE[k]

def _enrolment_calibration(ref_clip, ck, use_cache=True):
    """Return (ear_ref, model_mu, model_sd, n_ref, duration_s, physiology_baseline).

    The cache stores both the Stage-B model statistics and the physiological alert
    measurements used by the presentation table. Old cache entries without the physiology
    fields are rebuilt once rather than silently mixing definitions.
    """
    FE = _feats_of(ck)
    p = Path(ref_clip)
    key = f"{p.name}:{p.stat().st_size}"
    store = json.loads(ENROL_CACHE.read_text()) if ENROL_CACHE.exists() else {}
    if use_cache and key in store and "dur" in store[key] and "phys" in store[key]:
        e = store[key]
        phys = e.get("phys") or None
        if e["n"] < 4:
            return e["ear_ref"], None, None, e["n"], e["dur"], None
        return (e["ear_ref"], pd.Series(e["mu"], index=e["feats"])[FE],
                pd.Series(e["sd"], index=e["feats"])[FE], e["n"], e["dur"], phys)

    rraw = extract_video(ref_clip, ck["sample_every"], 20000, progress=False)
    rdf = prepare_video(rraw.assign(video_id="ref", subject="demo", state=0, split="demo"))
    ear_ref = float(rdf.attrs["ear_ref"])
    eff = float(rdf.eff_fps.iloc[0])
    dur = len(rdf) / eff
    rb = extract_blinks_for_video(rdf, eff)

    mu = sd = None
    phys = None
    if len(rb) >= 4:
        mu = rb[FE].mean()
        sd = rb[FE].std().replace(0.0, 1e-6)
        phys = _physiology_snapshot(rb, dur)

    store[key] = dict(
        ear_ref=ear_ref, n=int(len(rb)), dur=float(dur), feats=list(FE),
        mu=[] if mu is None else [float(mu[f]) for f in FE],
        sd=[] if sd is None else [float(sd[f]) for f in FE],
        phys=phys)
    ENROL_CACHE.write_text(json.dumps(store, indent=1))
    return ear_ref, mu, sd, len(rb), dur, phys

def _geometry_and_landmarks(frame_bgr, lm_engine):
    """frame_geometry's six features plus the landmarks, from one detect() call.

    frame_geometry does not return the landmarks and the live view needs them to draw. A
    second detect() would double the only expensive step, so the features are recomputed
    here from the same indices and helpers; the cell below checks they agree.
    """
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = lm_engine.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.face_landmarks:
        return None, None
    h, w = frame_bgr.shape[:2]
    lm = res.face_landmarks[0]
    P = lambda i: (lm[i].x * w, lm[i].y * h)
    ear = (_lm_ear([P(i) for i in LEFT_EYE]) + _lm_ear([P(i) for i in RIGHT_EYE])) / 2
    mar = _lm_mar([P(i) for i in MOUTH])
    head_drop = (P(1)[1] - (P(33)[1] + P(263)[1]) / 2) / h
    (pitch, yaw, roll), _, _, _ = head_pose(lm, w, h)
    return dict(ear=ear, mar=mar, head_drop=head_drop, pitch=pitch, yaw=yaw, roll=roll), lm

def _running_verdict(bands, scores):
    """Majority over completed sequences, mean as tie-break - the video_vote rule.

    On a tie the verdict is discretise(mean), which is not the label nearest the mean:
    at mean 7.5 discretise gives drowsy.
    """
    frac = [float(np.mean(np.asarray(bands, dtype=float) == c)) for c in LABELS_POOL]
    mx = max(frac)
    if sum(f == mx for f in frac) != 1:
        return float(discretise([float(np.mean(scores))])[0])
    return float(LABELS_POOL[int(np.argmax(frac))])

def _score_blinks(b, mu, sd, ck, model):
    """Stage B, Stage C and the forward pass - the tail of analyse_clip_blinks."""
    FE = _feats_of(ck)
    bc = b.copy()
    bc[FE] = (b[FE] - mu) / sd
    bc["video_id"] = "demo"; bc["subject"] = "demo"; bc["state"] = -1; bc["split"] = "demo"
    X, meta = unroll_blinks(bc, FE)
    X = ((X - np.array(ck["global_mu"])) / np.array(ck["global_sd"])).astype(np.float32)
    sc = predict_scores(model, X)
    return pd.DataFrame(dict(seq_i=meta.seq_i.values, score=sc, band=discretise(sc)))

def _live_analyse_prefix(rows, src_fps, ear_ref, mu, sd, ck, model):
    """Re-analyse the accumulated live prefix and return the current scientific state.

    ear_ref=None intentionally invokes prepare_video's own-prefix P90. This is the
    no-enrolment condition; reprocessing the whole prefix means earlier blink events are
    updated when that provisional reference changes.
    """
    raw = pd.DataFrame(rows)
    raw["src_fps"] = src_fps
    raw = raw.assign(video_id="demo", subject="demo", state=-1, split="demo")
    df = prepare_video(raw, ear_ref=ear_ref)
    eff = float(df.eff_fps.iloc[0])
    b = extract_blinks_for_video(df, eff)

    if len(b) >= int(ck["seq_len"]):
        sq = _score_blinks(b, mu, sd, ck, model)
    else:
        sq = pd.DataFrame(columns=["seq_i", "score", "band"])

    return df, b, sq, eff, float(df.attrs["ear_ref"])

# ---- how the frame reaches the browser -----------------------------------------
# Under "mjpeg" the frames leave over a single multipart connection served by the route
# in the app cell: one JPEG replaces the last, a slow client drops frames instead of
# queueing them, and the Gradio event stream carries only the panels. Under "inline"
# each frame is a base64 payload on the event stream, which needs no extra route and is
# the fallback if a tunnel will not pass a long-lived multipart response. One viewer at
# a time either way.
_LIVE_JPG = {"buf": None}
_LIVE_NEW = threading.Event()

def _jpeg(bgr):
    ok, buf = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), LIVE_JPEG_Q])
    return buf.tobytes() if ok else None

def _publish(bgr):
    f = _jpeg(bgr)
    if f:
        _LIVE_JPG["buf"] = f
        _LIVE_NEW.set()

def _img_html(bgr):
    f = _jpeg(bgr)
    return "" if not f else (
        f'<img src="data:image/jpeg;base64,{base64.b64encode(f).decode()}" '
        f'style="width:100%;border-radius:6px;background:#111">')

def _stream_html():
    return (f'<img src="/live.mjpg?t={time.time():.0f}" '
            f'style="width:100%;border-radius:6px;background:#111">')

def _mjpeg_stream(idle_stop_s=30.0):
    """Latest frame only. Ends once nothing has been published for idle_stop_s, leaving
    the last frame on screen."""
    idle = 0.0
    while idle < idle_stop_s:
        if _LIVE_NEW.wait(timeout=0.5):
            _LIVE_NEW.clear()
            idle = 0.0
            f = _LIVE_JPG["buf"]
            if f:
                yield (b"--frame\r\nContent-Type: image/jpeg\r\nContent-Length: "
                       + str(len(f)).encode() + b"\r\n\r\n" + f + b"\r\n")
        else:
            idle += 0.5

def _open_encoder(out_path, w, h, fps):
    """H.264 straight from the drawn frames, same settings as the 12.1 overlay."""
    if not (LIVE_WRITE_OVERLAY and shutil.which("ffmpeg")):
        return None, None
    size = (w // 2 * 2, h // 2 * 2)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    enc = subprocess.Popen(
        ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "bgr24",
         "-s", f"{size[0]}x{size[1]}", "-r", f"{fps:.6f}", "-i", "pipe:0",
         "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "veryfast",
         "-crf", str(OVERLAY_CRF), "-movflags", "+faststart", str(out_path)],
        stdin=subprocess.PIPE)
    return enc, size

_MESH_STEP = 6

def _spark(disp, hist, now_s):
    """Rolling calibrated EAR. A value of 1.0 is the active Stage-A open-eye reference.

    Under enrolment that reference is fixed from the alert clip. Without enrolment it is
    the running P90 of the footage seen so far and can move as the prefix grows.
    """
    if len(hist) < 2:
        return
    h, w = disp.shape[:2]
    x0, y0, bw, bh = 10, h - 66, w - 20, 54
    cv2.rectangle(disp, (x0, y0), (x0 + bw, y0 + bh), (18, 18, 22), -1)
    t = np.array([p[0] for p in hist])
    v = np.array([p[1] for p in hist])
    t0 = max(0.0, now_s - LIVE_SPARK_S)
    keep = t >= t0
    if keep.sum() < 2:
        return
    t, v = t[keep], np.clip(v[keep], 0.0, 1.35)
    xs = x0 + ((t - t0) / max(LIVE_SPARK_S, 1e-6) * bw).astype(np.int32)
    ys = (y0 + bh - (v / 1.35) * bh).astype(np.int32)
    yr = int(y0 + bh - (1.0 / 1.35) * bh)
    cv2.line(disp, (x0, yr), (x0 + bw, yr), (70, 90, 80), 1)
    cv2.polylines(disp, [np.stack([xs, ys], 1)], False, (90, 230, 170), 1, cv2.LINE_AA)
    cv2.putText(disp, f"ear_cal, last {LIVE_SPARK_S:.0f}s", (x0 + 6, y0 + 13),
                cv2.FONT_HERSHEY_SIMPLEX, 0.34, (150, 150, 150), 1, cv2.LINE_AA)

def _draw_live(disp, lm, hist, now_s, ear_cal, mar, n_blinks, need, blink_now,
               band_txt, score, calibration_label):
    """Draw the current frame and make the calibration condition visible on the video."""
    h, w = disp.shape[:2]
    if lm is None:
        cv2.putText(disp, "NO FACE", (14, 34), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (40, 40, 230), 2, cv2.LINE_AA)
    else:
        for p in lm[::_MESH_STEP]:
            cv2.circle(disp, (int(p.x * w), int(p.y * h)), 1, (130, 130, 130), -1)
        for idxs, col in ((LEFT_EYE, (90, 230, 170)), (RIGHT_EYE, (90, 230, 170)),
                          (MOUTH, (240, 180, 100))):
            pts = np.array([[int(lm[i].x * w), int(lm[i].y * h)] for i in idxs], np.int32)
            cv2.polylines(disp, [pts], True, col, 1, cv2.LINE_AA)

    fs = max(0.42, w / 1100.0)
    ear_txt = "--" if not np.isfinite(ear_cal) else f"{ear_cal:5.2f}"
    mar_txt = "--" if not np.isfinite(mar) else f"{mar:5.3f}"
    cal_col = (90, 230, 170) if calibration_label.startswith("IN PROTOCOL") else (80, 190, 240)
    rows = [
        ("MEASURED NOW", (150, 150, 150), fs * 0.72, 1),
        (f"ear_cal {ear_txt}   mar {mar_txt}", (235, 235, 235), fs, 1),
        (f"blinks  {n_blinks:3d}" + ("   BLINK" if blink_now else ""),
         (90, 230, 170) if blink_now else (235, 235, 235), fs, 1),
        None,
        ("CALIBRATION", (150, 150, 150), fs * 0.72, 1),
        (calibration_label, cal_col, fs * 0.78, 1),
        None,
        ("MODEL", (150, 150, 150), fs * 0.72, 1),
    ]
    rows.append((f"warming up  {min(n_blinks, need)}/{need} blinks",
                 (150, 190, 240), fs, 1)
                if score is None else
                (f"{band_txt}   {score:.1f}/10", (235, 235, 235), fs * 1.05, 2))
    _hud_block(disp, 12, 12, rows, colour=(18, 18, 22))
    _spark(disp, hist, now_s)

def _live_plot(ax, b, eff, sq_span, sq_score, sq_band, now_s, need, n_blinks,
               base_rate, warm_end):
    """Blink rate, completed sequence scores, and the majority vote.

    The dashed blink-rate reference is shown only when a usable personal alert enrolment
    exists. No population statistic is drawn as if it were this driver's physiology.
    """
    for a in ax:
        a.clear()
    span = max(now_s, 5.0)

    bt = (b.bottom_f.values / eff) if n_blinks else np.array([])
    grid = np.arange(0.0, span + 1e-9, 2.0)
    rate = [len(bt[(bt > g - 60) & (bt <= g)]) / max(min(60.0, g), 1e-6) * 60
            for g in grid]
    ax[0].plot(grid, rate, lw=1.4, color="#2F6F5B")
    if base_rate is not None and np.isfinite(base_rate):
        ax[0].axhline(base_rate, color="#888", ls="--", lw=1)
        ax[0].text(span, base_rate, f"  personal alert baseline {base_rate:.1f}/min",
                   va="center", fontsize=8, color="#666")
    ax[0].set_ylabel("blinks / min")
    ax[0].set_ylim(0, max(28.0, (max(rate) if len(rate) else 0) * 1.15))
    ax[0].set_title(f"{n_blinks} blinks retrieved", fontsize=10, loc="left")

    ax[1].fill_between([0, span], 0, CUT_LO, color="#9FE1CB", alpha=.22)
    ax[1].fill_between([0, span], CUT_LO, CUT_HI, color="#E8D9A8", alpha=.28)
    ax[1].fill_between([0, span], CUT_HI, 10, color="#F5C4B3", alpha=.30)
    for y, lab, c in ((CUT_LO / 2, "alert", 0.0),
                      ((CUT_LO + CUT_HI) / 2, "low vig.", 5.0),
                      ((CUT_HI + 10) / 2, "drowsy", 10.0)):
        ax[1].text(span, y, "  " + lab, va="center", fontsize=8, color=_BANDC[c])
    if warm_end is not None:
        ax[1].axvspan(0, warm_end, color="#999", alpha=.13)
        ax[1].text(warm_end / 2, 9.3, f"warm-up - no verdict until blink {need}",
                   ha="center", fontsize=8, color="#666", style="italic")

    if len(sq_score):
        ends = [s1 for _, s1 in sq_span]
        for (s0, s1), sc, bd in zip(sq_span, sq_score, sq_band):
            ax[1].plot([s0, s1], [sc, sc], lw=2.5, alpha=.18, color=_BANDC[bd],
                       solid_capstyle="butt")
        run = np.cumsum(sq_score) / np.arange(1, len(sq_score) + 1)
        ax[1].step(ends + [now_s], list(run) + [run[-1]], where="post", lw=1.4,
                   color="#1A1D24", alpha=.85, zorder=3)
        for s1, sc, bd in zip(ends, sq_score, sq_band):
            ax[1].plot(s1, sc, "o", ms=5, color=_BANDC[bd], zorder=4)
        ax[1].text(now_s, run[-1], f"  mean {run[-1]:.1f}", va="center", fontsize=8,
                   color="#1A1D24")
    else:
        ax[1].text(span / 2, 5.0, f"no sequence yet - {need} blink events must accumulate",
                   ha="center", va="center", fontsize=10, color="#666")
    ax[1].axvline(now_s, color="#B33A31", lw=1.2, alpha=.75)
    ax[1].set_ylim(0, 10)
    ax[1].set_ylabel("sequence score")
    ax[1].set_xlabel("seconds processed   -   bar = footage consumed by one 30-blink sequence")

    left, tot = 0.0, max(len(sq_band), 1)
    for c in (0.0, 5.0, 10.0):
        n = int(np.sum(np.asarray(sq_band) == c)) if len(sq_band) else 0
        if n:
            ax[2].barh(0, n / tot, left=left, color=_BANDC[c], height=.6)
            ax[2].text(left + n / tot / 2, 0, str(n), ha="center", va="center",
                       fontsize=8, color="white")
            left += n / tot
    ax[2].set_xlim(0, 1)
    ax[2].set_ylim(-.5, .5)
    ax[2].set_yticks([])
    ax[2].set_xticks([])
    ax[2].set_xlabel("vote across completed sequences", fontsize=8)
    ax[0].set_xlim(0, span)
    ax[1].set_xlim(0, span)

def _blank(msg):
    """A message with every other output left alone."""
    return (msg,) + tuple(_SKIP() for _ in range(7))

def g_live(choice, upload, ref_upload, use_enrolment):
    path = upload or _all_choices.get(choice)
    if not path:
        yield _blank("Pick a clip, upload one, or record from your webcam.")
        return

    auto_ref = _auto_enrolment(path) if use_enrolment else None
    ref = ref_upload or auto_ref
    FE = _feats_of(_g_ck)

    # Three calibration states mirror analyse_clip_blinks exactly.
    if ref is not None:
        yield _blank("### Reading the alert reference")
        ear_ref, mu, sd, n_ref, ref_dur, phys_base = _enrolment_calibration(ref, _g_ck)
        if mu is None:
            mode = "partial"
            mu = pd.Series(_g_ck["fallback_mu"], index=FE)
            sd = pd.Series(_g_ck["fallback_sd"], index=FE)
            phys_base = None
            base_rate = None
            yield _blank(
                f"### Continuing with a Stage-B fallback\n\nThe alert reference produced "
                f"only **{n_ref} blink{'s' if n_ref != 1 else ''}**. Stage A can still use "
                f"its open-eye reference, but Stage B needs at least four blinks, so the "
                f"population alert statistics will be used. This run is **out of protocol**.")
        else:
            mode = "enrolled"
            base_rate = phys_base["rate_pm"] if phys_base else None
    else:
        mode = "fallback"
        ear_ref = None                       # prepare_video computes the prefix P90
        mu = pd.Series(_g_ck["fallback_mu"], index=FE)
        sd = pd.Series(_g_ck["fallback_sd"], index=FE)
        n_ref, ref_dur, phys_base, base_rate = 0, None, None, None
        yield _blank(
            "### No enrolment - running the out-of-protocol comparison\n\n"
            "Stage A will use the **running P90 of the footage seen so far** and Stage B "
            "will use **population alert statistics**. The P90 is recomputed over the "
            "whole prefix every analysis update; at the end it becomes the same full-clip "
            "P90 used by Summary.")

    yield _blank("### Opening the clip")
    need = int(_g_ck["seq_len"])
    rot = _cached_rotation(path)
    lm_engine = thread_landmarker()
    cap = cv2.VideoCapture(str(path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    every = int(_g_ck["sample_every"])
    eff_nominal = src_fps / every
    analyse_every = max(1, int(round(eff_nominal * LIVE_ANALYSE_S)))

    fig, ax = plt.subplots(3, 1, figsize=(11, 5.6),
                           gridspec_kw=dict(height_ratios=[1.0, 1.7, 0.22]))
    fig.subplots_adjust(left=.08, right=.90, top=.94, bottom=.11, hspace=.42)

    rows, idx, kept, hist = [], 0, 0, []
    sq_span, sq_score, sq_band = [], [], []
    n_blinks, band_txt, score_now, last_end, warm_end = 0, "", None, -1.0, None
    ear_now, mar_now, blink_now, now_s = float("nan"), float("nan"), False, 0.0
    msg, tbl, fresh = "### Starting", pd.DataFrame(), False
    active_ear_ref = ear_ref
    t_lm = t_an = t_dr = 0.0
    t_wall = time.perf_counter()
    last_pub = last_emit = 0.0
    last_analysed_kept = 0
    enc, enc_size = None, None

    mode_slug = {"enrolled": "enrolled", "partial": "stageB_fallback",
                 "fallback": "no_enrolment"}[mode]
    live_over = _ANNOT_DIR / f"live_{mode_slug}_{Path(path).stem}.mp4"

    def _cal_label():
        if mode == "enrolled":
            return "IN PROTOCOL - PERSONALISED"
        if mode == "partial":
            return "OUT OF PROTOCOL - STAGE-B FALLBACK"
        return "OUT OF PROTOCOL - NO ENROLMENT"

    def _make_message():
        if score_now is None:
            filled = min(need, n_blinks)
            bar = "█" * filled + "░" * (need - filled)
            out = (f"### Warming up\n\n**{now_s:,.0f}s processed**\n\n"
                   f"`{bar}`  {n_blinks}/{need} blink events\n\n"
                   f"**No verdict yet.** A complete model input needs {need} consecutive "
                   f"blink events.")
        else:
            v = [int(np.sum(np.asarray(sq_band) == c)) for c in (0.0, 5.0, 10.0)]
            out = (f"### {band_txt} &nbsp;·&nbsp; {score_now:.1f}/10\n\n"
                   f"**{now_s:,.0f}s processed** &nbsp;·&nbsp; "
                   f"{len(sq_score)} sequence{'' if len(sq_score) == 1 else 's'} "
                   f"&nbsp;·&nbsp; {n_blinks} blinks\n\n"
                   f"Vote: {v[0]} alert / {v[1]} low vigilance / {v[2]} drowsy. "
                   f"The verdict is the majority, with the mean as tie-break.")

        if mode == "enrolled":
            out += (f"\n\n<sub>&#9989; **In protocol.** Stage A and Stage B use this "
                    f"driver's alert enrolment ({n_ref} reference blinks; open-eye P90 "
                    f"{active_ear_ref:.4f}).</sub>")
        elif mode == "partial":
            out += (f"\n\n<sub>&#9888;&#65039; **Out of protocol.** Stage A uses the "
                    f"supplied alert reference (open-eye P90 {active_ear_ref:.4f}), but "
                    f"Stage B uses population alert statistics because the reference "
                    f"contained only {n_ref} blinks.</sub>")
        else:
            ref_txt = "not stable yet" if active_ear_ref is None else f"{active_ear_ref:.4f}"
            out += (f"\n\n<sub>&#9888;&#65039; **Out of protocol - no enrolment.** "
                    f"Stage A running self-reference P90: **{ref_txt}**. Stage B uses "
                    f"population alert statistics. The self-reference can move as more "
                    f"footage arrives.</sub>")
        return out

    def _analyse_now():
        nonlocal active_ear_ref, n_blinks, blink_now, last_end, warm_end
        nonlocal sq_span, sq_score, sq_band, score_now, band_txt, now_s, tbl, hist
        nonlocal last_analysed_kept, t_an
        _t = time.perf_counter()
        df, b, sq, eff, active_ear_ref = _live_analyse_prefix(
            rows, src_fps, ear_ref if mode != "fallback" else None,
            mu, sd, _g_ck, _g_model)
        last_analysed_kept = kept
        now_s = len(df) / eff
        n_blinks = len(b)

        new_last_end = float(b.end_f.iloc[-1] / eff) if n_blinks else -1.0
        blink_now = bool(n_blinks and new_last_end > last_end + 1e-9)
        last_end = max(last_end, new_last_end)

        tbl = _physiology_table(b, now_s, baseline=phys_base, current_label="clip so far")
        warm_end = (float(b.sort_values("start_f").start_s.values[need - 1])
                    if n_blinks >= need else None)

        if len(sq):
            st = b.sort_values("start_f").start_s.values
            sq_span = [(float(st[min(len(st) - 1, i * BLINK_SEQ_STRIDE)]),
                        float(st[min(len(st) - 1,
                                     i * BLINK_SEQ_STRIDE + need - 1)]))
                       for i in sq.seq_i]
            sq_score = list(sq.score.values)
            sq_band = list(sq.band.values)
            score_now = float(np.mean(sq_score))
            band_txt = _BAND[_running_verdict(sq_band, sq_score)]
        else:
            sq_span, sq_score, sq_band = [], [], []
            score_now, band_txt = None, ""

        # Rebuild the spark from the current Stage-A reference. This matters in fallback
        # mode because the prefix P90 can change and old normalised values must not remain.
        n_hist = max(2, int(round((LIVE_SPARK_S + LIVE_ANALYSE_S) * eff)))
        i0 = max(0, len(df) - n_hist)
        hist = [(float(i / eff), float(df.ear_cal.iloc[i]))
                for i in range(i0, len(df)) if np.isfinite(df.ear_cal.iloc[i])]

        _live_plot(ax, b, eff, sq_span, sq_score, sq_band, now_s, need,
                   n_blinks, base_rate, warm_end)
        t_an += time.perf_counter() - _t
        return df, b, sq, eff

    _LIVE_JPG["buf"] = None
    if LIVE_TRANSPORT == "mjpeg":
        yield (_SKIP(), _stream_html()) + tuple(_SKIP() for _ in range(6))

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if idx % every:
                idx += 1
                continue
            idx += 1
            if rot is not None:
                frame = cv2.rotate(frame, rot)

            det = frame
            if LIVE_DETECT_W and det.shape[1] > LIVE_DETECT_W:
                det = cv2.resize(det, (LIVE_DETECT_W, int(round(
                    det.shape[0] * LIVE_DETECT_W / det.shape[1]))))
            _t = time.perf_counter()
            g, lms = _geometry_and_landmarks(det, lm_engine)
            t_lm += time.perf_counter() - _t

            rows.append(dict(frame=kept, face=0 if g is None else 1,
                             **(dict(ear=np.nan, mar=np.nan, head_drop=np.nan,
                                     pitch=np.nan, yaw=np.nan, roll=np.nan)
                                if g is None else g)))
            kept += 1
            now_s = kept / eff_nominal

            if g is not None:
                mar_now = g["mar"]
                if active_ear_ref is not None and np.isfinite(active_ear_ref):
                    ear_now = g["ear"] / active_ear_ref
                    hist.append((now_s, ear_now))
                else:
                    ear_now = float("nan")
            else:
                ear_now, mar_now = float("nan"), float("nan")

            if kept % analyse_every == 0:
                _analyse_now()
                if g is not None and active_ear_ref is not None:
                    ear_now = g["ear"] / active_ear_ref
                msg = _make_message()
                fresh = True

            _t = time.perf_counter()
            disp = frame
            if disp.shape[1] > LIVE_DRAW_W:
                disp = cv2.resize(disp, (LIVE_DRAW_W, int(round(
                    disp.shape[0] * LIVE_DRAW_W / disp.shape[1]))))
            else:
                disp = disp.copy()
            _draw_live(disp, lms, hist, now_s, ear_now, mar_now, n_blinks, need,
                       blink_now, band_txt, score_now, _cal_label())
            small = cv2.resize(disp, (LIVE_VIEW_W, int(round(
                disp.shape[0] * LIVE_VIEW_W / disp.shape[1]))))

            if enc is None:
                enc, enc_size = _open_encoder(live_over, disp.shape[1], disp.shape[0],
                                              eff_nominal)
            if enc is not None:
                enc.stdin.write(disp[:enc_size[1], :enc_size[0]].tobytes())

            _now = time.perf_counter()
            if LIVE_TRANSPORT == "mjpeg" and (_now - last_pub) >= 1.0 / LIVE_PUBLISH_HZ:
                _publish(small)
                last_pub = _now
            t_dr += time.perf_counter() - _t

            if LIVE_REALTIME:
                _delay = (t_wall + now_s) - time.perf_counter()
                if _delay > 0:
                    time.sleep(_delay)
            _now = time.perf_counter()

            send_img = (LIVE_TRANSPORT == "inline"
                        and (fresh or (_now - last_emit) >= 1.0 / LIVE_EMIT_HZ))
            if send_img:
                last_emit = _now
            if fresh:
                yield ((msg, _img_html(small) if send_img else _SKIP(), fig, tbl)
                       + tuple(_SKIP() for _ in range(4)))
                fresh = False
            elif send_img:
                yield (_SKIP(), _img_html(small)) + tuple(_SKIP() for _ in range(6))
    finally:
        cap.release()
        if enc is not None:
            enc.stdin.close()
            enc.wait()

    if kept == 0:
        yield _blank("### Could not read any frames from this clip.")
        return

    # Always analyse the complete buffered clip once more. This removes the old edge case
    # where the final 0-2 seconds were omitted if the file ended between analysis ticks,
    # and makes fallback Live converge exactly to Summary's full-clip P90 condition.
    if last_analysed_kept != kept:
        _analyse_now()
        msg = _make_message()

    elapsed = time.perf_counter() - t_wall
    per = lambda x: x / max(kept, 1) * 1000
    timing = (f"<sub>{kept} sampled frames of {now_s:,.0f}s footage in "
              f"{elapsed:,.0f}s wall clock ({now_s / max(elapsed, 1e-6):.2f}x real time) "
              f"- landmarking {per(t_lm):.0f} ms/frame, drawing/encoding {per(t_dr):.0f} "
              f"ms/frame, prefix analysis {t_an:.1f}s total.</sub>")

    # A pre-rendered §12.1 overlay is safe only for the normal automatic-enrolment path.
    # Never show it after a fallback run: its burned-in verdict may use different calibration.
    # Also do not reuse an old live_*.mp4 unless this run actually wrote it.
    wrote_live_overlay = enc is not None
    pre = _ANNOT_DIR / f"annotated_{Path(path).stem}.mp4"
    pre_matches = (mode == "enrolled" and ref_upload is None and use_enrolment
                   and ref == auto_ref)
    over = None
    if wrote_live_overlay and live_over.exists() and live_over.stat().st_size > 50_000:
        over = live_over
    elif pre_matches and pre.exists() and pre.stat().st_size > 50_000:
        over = pre

    overlay_note = (" A matching rendered overlay has been loaded."
                    if over else
                    " Use **Show annotated video** if you want a rendered overlay for "
                    "this exact calibration condition.")
    final = (msg + "\n\n---\n\n**Clip finished.** The Summary tab is filled in."
             + overlay_note + "\n\n" + timing)

    protocol_text = {
        "enrolled": (f"**In protocol.** Stage A and Stage B use this driver's alert "
                     f"enrolment ({n_ref} reference blinks; P90 {active_ear_ref:.4f})."),
        "partial": (f"**Out of protocol.** Stage A uses the supplied alert reference, "
                    f"but Stage B uses population statistics because only {n_ref} "
                    f"reference blinks were retrieved."),
        "fallback": (f"**Out of protocol - no enrolment.** Final Stage-A self-reference "
                     f"P90 is {active_ear_ref:.4f}; Stage B uses population alert "
                     f"statistics. This final P90 is the same whole-clip reference used "
                     f"by Summary.")
    }[mode]

    summary = (f"## {band_txt or 'No verdict'}"
               + (f" &nbsp;·&nbsp; {score_now:.1f}/10" if score_now is not None else "")
               + f"\n\n**{Path(path).stem}** &nbsp;·&nbsp; {len(sq_score)} sequences "
                 f"over {n_blinks} blinks\n\n{protocol_text}")

    yield (final, _SKIP(), fig, tbl, summary, fig, tbl,
           str(over) if over else _SKIP())


In [ ]:
# ---- optional pre-cache ---------------------------------------------------------
# Pre-caching every enrolment clip is useful before a presentation, but it is expensive and
# should not block every notebook startup. A selected reference is cached automatically the
# first time live scoring uses it.
PRECACHE_ALL_ENROLMENTS = False

if PRECACHE_ALL_ENROLMENTS:
    _enrol_clips = sorted(DEMO_DIR.glob("s*_enrol_*.mp4"))
    try:
        _cached = json.loads(ENROL_CACHE.read_text()) if ENROL_CACHE.exists() else {}
    except Exception:
        print("enrolment_cache.json was unreadable; rebuilding it")
        _cached = {}
    _todo = [p for p in _enrol_clips if f"{p.name}:{p.stat().st_size}" not in _cached]
    print(f"{len(_enrol_clips)} enrolment clips, {len(_enrol_clips) - len(_todo)} already cached")
    for _p in tqdm(_todo, desc="enrolment", unit="clip", disable=not _todo):
        _enrolment_calibration(_p, _g_ck)
    _store = json.loads(ENROL_CACHE.read_text()) if ENROL_CACHE.exists() else {}
    _usable = sum(1 for v in _store.values() if v.get("n", 0) >= 4)
    print(f"cache: {len(_store)} clips, {_usable} usable for Stage B -> {ENROL_CACHE.name}")
else:
    print("full enrolment pre-cache skipped; selected references cache on first use")

# Orientation probing is cheap enough to do lazily as well.
print("clip orientation will be detected and cached on first use")


full enrolment pre-cache skipped; selected references cache on first use
clip orientation will be detected and cached on first use


In [ ]:
# ---- lightweight live-path check ------------------------------------------------
# Two detections verify that the drawing helper and batch helper compute the same geometry.
_cap = cv2.VideoCapture(str(_demo[0]))
_ok, _fr = _cap.read(); _cap.release()
if not _ok:
    print("could not read first demo frame - live geometry check skipped")
else:
    _lmx = new_landmarker()
    _a = frame_geometry(_fr, _lmx)
    _b2, _lms = _geometry_and_landmarks(_fr, _lmx)
    if _a is None or _b2 is None:
        print("no face on the first frame - live geometry check skipped")
    else:
        _diff = {k: abs(_a[k] - _b2[k]) for k in _a}
        assert max(_diff.values()) < 1e-6, _diff
        print(f"live geometry matches frame_geometry (max diff {max(_diff.values()):.2e}); "
              f"{len(_lms)} landmarks available for drawing")

RUN_SPEED_DIAGNOSTIC = False
if RUN_SPEED_DIAGNOSTIC:
    _W = 480
    _cap = cv2.VideoCapture(str(_demo[0]))
    _d_ear, _t_full, _t_small, _n = [], 0.0, 0.0, 0
    for _i in range(150):
        _ok, _f = _cap.read()
        if not _ok:
            break
        if _i % 5:
            continue
        _t0 = time.perf_counter(); _gf = frame_geometry(_f, _lmx); _t_full += time.perf_counter() - _t0
        _s = cv2.resize(_f, (_W, int(round(_f.shape[0] * _W / _f.shape[1])))) if _f.shape[1] > _W else _f
        _t0 = time.perf_counter(); _gs = frame_geometry(_s, _lmx); _t_small += time.perf_counter() - _t0
        if _gf and _gs:
            _d_ear.append(abs(_gf["ear"] - _gs["ear"])); _n += 1
    _cap.release()
    if _n:
        print(f"detection width: full vs {_W}px over {_n} frames")
        print(f"EAR difference median {np.median(_d_ear):.5f}, max {max(_d_ear):.5f}")
else:
    print("speed diagnostic skipped (set RUN_SPEED_DIAGNOSTIC = True to run it)")


live geometry matches frame_geometry (max diff 0.00e+00); 478 landmarks available for drawing
speed diagnostic skipped (set RUN_SPEED_DIAGNOSTIC = True to run it)


In [ ]:
# ---- the app ------------------------------------------------------------------
# The frame route is added to Gradio's own FastAPI app after launch and moved to the front
# of the router, because routes are matched in order and the front end registers a
# catch-all. Under LIVE_TRANSPORT = "inline" the route is unused and can fail silently.
from fastapi.responses import StreamingResponse

if "demo_app" in globals():
    try:
        demo_app.close()
    except Exception:
        pass

with gr.Blocks(title="Driver fatigue - blink-sequence model",
               theme=gr.themes.Soft(primary_hue="teal")) as demo_app:
    gr.Markdown(
        "# Driver fatigue from blink sequences\n"
        "A regression model over sequences of **30 blink events**, reimplementing the "
        "UTA-RLDD baseline (Ghoddoosian et al., 2019) under a stricter evaluation "
        "protocol. Every demo clip comes from a **held-out test subject**.\n\n"
        "The model never sees pixels. It sees four numbers per blink - duration, "
        "amplitude, eye-opening velocity and frequency - and judges the sequence.")

    with gr.Row():
        with gr.Column(scale=1, min_width=260):
            gr.Markdown("### Input")
            dd = gr.Dropdown(choices=list(_all_choices.keys()),
                             label="Demo clip (held-out subjects)")
            up = gr.Video(sources=["upload", "webcam"],
                          label="or upload / record your own")
            gr.Markdown("### Calibration")
            enr = gr.Checkbox(value=True,
                              label="Use this driver's enrolment clip when one exists")
            ref = gr.Video(sources=["upload", "webcam"],
                           label="Alert reference clip (overrides the above)")
            gr.Markdown(
                "<sub>**Why enrolment matters.** The published method z-scores each "
                "driver's blinks against their own alert baseline, so the system needs a "
                "reference recording of that person awake. A car does not have one at the "
                "start of a trip - and assuming the driver is alert then fails precisely "
                "in the case the system exists to catch. **Untick the box to compare that "
                "fallback live on the same clip.**</sub>")

        with gr.Column(scale=3):
            with gr.Tabs():
                with gr.Tab("Live"):
                    gr.Markdown(
                        "Scores the clip **as it is read**, frame by frame. Nothing is "
                        "precomputed: the trace, the blink counter, the feature table and "
                        "the score appear as the evidence arrives. With enrolment it uses "
                        "the driver's fixed alert baseline. Untick enrolment to demonstrate "
                        "the out-of-protocol condition live: running self-P90 for Stage A "
                        "plus population Stage-B statistics. At clip end that self-P90 "
                        "matches the whole-clip Summary condition.")
                    lv_btn = gr.Button("Start live scoring", variant="primary")
                    with gr.Row():
                        lv_img = gr.HTML()
                        lv_md = gr.Markdown()
                    lv_plot = gr.Plot(label="Blink rate, the score per sequence, and the vote")
                    lv_tbl = gr.Dataframe(
                        label="Physiology so far - dissertation validation quantities",
                        wrap=True)

                with gr.Tab("Summary"):
                    gr.Markdown(
                        "Scores the whole clip first, then reports. Works without "
                        "enrolment, and labels the result out of protocol when it is.")
                    go = gr.Button("Analyse whole clip", variant="primary")
                    verdict_md = gr.Markdown()
                    feat_out = gr.Dataframe(
                        label="Physiology - same quantities used in the dissertation checks",
                        wrap=True)
                    tl_plot = gr.Plot(label="Calibrated EAR with retrieved blinks, and "
                                            "the score per sequence")

                with gr.Tab("Rendered overlay"):
                    gr.Markdown(
                        "The overlay for this clip: mesh, eye and mouth contours, live "
                        "features and the band. A matching pre-rendered file is reused only "
                        "when its calibration matches the current controls. Otherwise click "
                        "**Show annotated video** to render this exact enrolment/fallback "
                        "condition; a stale overlay is never substituted.")
                    rv_btn = gr.Button("Show annotated video")
                    rv_vid = gr.Video(label="Annotated clip")

    go.click(g_analyse, [dd, up, ref, enr], [verdict_md, tl_plot, feat_out])
    rv_btn.click(g_annotated, [dd, up, ref, enr], rv_vid)
    lv_btn.click(g_live, [dd, up, ref, enr],
                 [lv_md, lv_img, lv_plot, lv_tbl,
                  verdict_md, tl_plot, feat_out, rv_vid])

demo_app.queue()
_gr_app, _local_url, _share_url = demo_app.launch(
    share=True, debug=False, prevent_thread_lock=True,
    allowed_paths=[str(_DEMO_DIR)])

if LIVE_TRANSPORT == "mjpeg":
    @_gr_app.get("/live.mjpg")
    def _live_mjpg():
        return StreamingResponse(_mjpeg_stream(),
                                 media_type="multipart/x-mixed-replace; boundary=frame",
                                 headers={"Cache-Control": "no-store"})
    _gr_app.router.routes.insert(0, _gr_app.router.routes.pop())

print(f"local : {_local_url}")
print(f"share : {_share_url}")
if LIVE_TRANSPORT == "mjpeg":
    print(f"frames: {_share_url}live.mjpg")
else:
    print("frames: inline Gradio transport (no custom MJPEG route)")
print("The server runs in the background. Re-running this cell restarts it.")

/tmp/ipykernel_2388/3018387274.py:13: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Driver fatigue - blink-sequence model",


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a28d740050bc53c7c1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


local : http://127.0.0.1:7860/
share : https://a28d740050bc53c7c1.gradio.live
frames: inline Gradio transport (no custom MJPEG route)
The server runs in the background. Re-running this cell restarts it.
